In [21]:
import sys
sys.path.insert(0, r'C:\projects\hedge_fund')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client

db = DuckDBClient()
s  = S3Client()

# Load all clean USDINR data (both contracts)
all_files  = s.list_files('processed/1s/USDINR26SEPFUT/')
all_files += s.list_files('processed/1s/USDINR26OCTFUT/')

all_df = []
for f in all_files:
    parts = f.split('/')
    sym   = parts[2]
    date  = parts[3].replace('.parquet','')
    try:
        df = db.read_parquet(f'processed/1s/{sym}/{date}.parquet')
        df['ist_sec'] = (df['ts_sec'] + 19800) % 86400
        df = df[(df['ist_sec'] >= 9*3600) &
                (df['ist_sec'] <= 17*3600)].reset_index(drop=True)
        df['date']   = date
        df['symbol'] = sym
        all_df.append(df)
        print(f'{sym}/{date}: {len(df)} bars')
    except Exception as e:
        print(f'ERROR {sym}/{date}: {e}')

db.close()
df_raw = pd.concat(all_df, ignore_index=True).reset_index(drop=True)
print(f'\nTotal: {len(df_raw):,} bars across {df_raw["date"].nunique()} days')
print(f'Dates: {sorted(df_raw["date"].unique())}')
print(f'Columns: {len(df_raw.columns)}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26SEPFUT/2026-09-22: 12568 bars


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26SEPFUT/2026-09-23: 11031 bars
USDINR26SEPFUT/2026-09-24: 10385 bars


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26SEPFUT/2026-09-25: 15469 bars
USDINR26SEPFUT/2026-09-28: 2921 bars


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26OCTFUT/2026-09-30: 16701 bars


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26OCTFUT/2026-10-01: 16554 bars

Total: 85,629 bars across 7 days
Dates: ['2026-09-22', '2026-09-23', '2026-09-24', '2026-09-25', '2026-09-28', '2026-09-30', '2026-10-01']
Columns: 55


In [22]:
# ── Layer 2: Feature Engineering ──────────────────────────────────
# Rules:
#   1. Compute per day — no cross-day bleeding
#   2. No lookahead bias
#   3. Every feature normalized within its own rolling window

def build_features(df_raw):
    feature_dfs = []
    
    for d in sorted(df_raw['date'].unique()):
        sub = df_raw[df_raw['date']==d].copy().reset_index(drop=True)
        
        # ── Returns ───────────────────────────────────────────────
        sub['ret']     = sub['close'].pct_change()
        sub['log_ret'] = np.log(sub['close'] / sub['close'].shift(1))
        
        # ── Realized Volatility (term structure) ──────────────────
        for w in [10, 30, 60, 120, 300, 600, 1800]:
            sub[f'rv_{w}'] = sub['ret'].rolling(w).std() * np.sqrt(w)
        
        # Vol ratios (slope of term structure)
        sub['rv_10_30']    = sub['rv_10']   / (sub['rv_30']   + 1e-9)
        sub['rv_30_60']    = sub['rv_30']   / (sub['rv_60']   + 1e-9)
        sub['rv_60_300']   = sub['rv_60']   / (sub['rv_300']  + 1e-9)
        sub['rv_300_1800'] = sub['rv_300']  / (sub['rv_1800'] + 1e-9)
        
        # Vol direction
        sub['rv_accel']  = sub['rv_10']  - sub['rv_60']
        sub['rv_trend']  = sub['rv_60']  - sub['rv_300']
        sub['rv_trend2'] = sub['rv_300'] - sub['rv_1800']
        
        # ── Spread ────────────────────────────────────────────────
        sub['spread']     = sub['ask_p1'] - sub['bid_p1']
        sub['spread_bps'] = sub['spread'] / (sub['close'] + 1e-9) * 10000
        
        for w in [30, 60, 300, 600, 1800]:
            mu = sub['spread'].rolling(w).mean()
            sd = sub['spread'].rolling(w).std() + 1e-9
            sub[f'spread_z_{w}']  = (sub['spread'] - mu) / sd
            sub[f'spread_ma_{w}'] = mu
        
        sub['spread_trend_s'] = sub['spread_ma_30']  - sub['spread_ma_300']
        sub['spread_trend_l'] = sub['spread_ma_300'] - sub['spread_ma_1800']
        sub['spread_accel']   = sub['spread_ma_30']  - sub['spread_ma_60']
        
        # ── Order Flow Imbalance (5 levels) ───────────────────────
        sub['ofi_1'] = sub['bid_q1'] - sub['ask_q1']
        sub['ofi_5'] = sum(sub[f'bid_q{i}'] - sub[f'ask_q{i}']
                           for i in range(1, 6))
        total_qty        = sub['total_bid_qty'] + sub['total_ask_qty'] + 1
        sub['ofi_norm']  = sub['ofi_5'] / total_qty
        sub['ofi_1_norm']= sub['ofi_1'] / total_qty
        
        for w in [10, 30, 60, 120, 300, 600]:
            sub[f'ofi_ma_{w}'] = sub['ofi_norm'].rolling(w).mean()
        
        sub['ofi_trend_s'] = sub['ofi_ma_30']  - sub['ofi_ma_300']
        sub['ofi_trend_l'] = sub['ofi_ma_60']  - sub['ofi_ma_600']
        sub['ofi_accel']   = sub['ofi_ma_10']  - sub['ofi_ma_60']
        
        # ── Cancel Imbalance (PRIMARY SIGNAL) ─────────────────────
        sub['bid_chg'] = sub['total_bid_qty'].diff()
        sub['ask_chg'] = sub['total_ask_qty'].diff()
        pu = sub['close'].diff().abs() < 0.0025 * 0.5
        sub['cancel_imb'] = 0.0
        sub.loc[pu & (sub['bid_chg'] < 0), 'cancel_imb'] -= sub['bid_chg'].abs()
        sub.loc[pu & (sub['ask_chg'] < 0), 'cancel_imb'] += sub['ask_chg'].abs()
        
        for w in [10, 30, 60, 120, 300, 600]:
            mu = sub['cancel_imb'].rolling(w).mean()
            sd = sub['cancel_imb'].rolling(w).std() + 1e-9
            sub[f'cancel_ma_{w}'] = mu
            sub[f'cancel_z_{w}']  = mu / sd
        
        sub['cancel_trend_s'] = sub['cancel_ma_30']  - sub['cancel_ma_300']
        sub['cancel_trend_l'] = sub['cancel_ma_60']  - sub['cancel_ma_600']
        sub['cancel_accel']   = sub['cancel_ma_10']  - sub['cancel_ma_60']
        
        # ── Weighted Mid ──────────────────────────────────────────
        sub['mid']      = (sub['bid_p1'] + sub['ask_p1']) / 2
        sub['wmid_dev'] = sub['weighted_mid'] - sub['mid']
        
        for w in [10, 30, 60, 120, 300]:
            mu = sub['wmid_dev'].rolling(w).mean()
            sd = sub['wmid_dev'].rolling(w).std() + 1e-9
            sub[f'wmid_ma_{w}'] = mu
            sub[f'wmid_z_{w}']  = sub['wmid_dev'] / sd
        
        sub['wmid_trend'] = sub['wmid_ma_30']  - sub['wmid_ma_300']
        sub['wmid_accel'] = sub['wmid_ma_10']  - sub['wmid_ma_60']
        
        # ── Price Momentum (multiple scales) ──────────────────────
        for w in [5, 10, 20, 30, 60, 120, 300, 600, 1800]:
            sub[f'mom_{w}'] = sub['close'].pct_change(w)
        
        sub['mom_s_m']   = sub['mom_10']  - sub['mom_60']
        sub['mom_m_l']   = sub['mom_60']  - sub['mom_300']
        sub['mom_l_vl']  = sub['mom_300'] - sub['mom_1800']
        
        # ── VWAP Deviation ────────────────────────────────────────
        sub['vwap_dev'] = (sub['close'] - sub['vwap']) / (sub['vwap'] + 1e-9) * 10000
        for w in [60, 300, 1800]:
            sd = sub['vwap_dev'].rolling(w).std() + 1e-9
            sub[f'vwap_z_{w}'] = sub['vwap_dev'] / sd
        
        # ── Volume ────────────────────────────────────────────────
        for w in [30, 60, 300, 1800]:
            mu = sub['tick_count'].rolling(w).mean() + 1e-9
            sub[f'vol_ratio_{w}'] = sub['tick_count'] / mu
        
        sub['vol_accel'] = sub['vol_ratio_30']  - sub['vol_ratio_300']
        sub['vol_trend'] = sub['vol_ratio_60']  - sub['vol_ratio_1800']
        
        # ── OI ────────────────────────────────────────────────────
        sub['oi_chg'] = sub['oi'].diff()
        for w in [30, 60, 300, 1800]:
            sd = sub['oi_chg'].rolling(w).std() + 1e-9
            sub[f'oi_z_{w}'] = sub['oi_chg'] / sd
        
        sub['oi_trend_s'] = sub['oi_z_30']  - sub['oi_z_300']
        sub['oi_trend_l'] = sub['oi_z_300'] - sub['oi_z_1800']
        
        # ── Price Range ───────────────────────────────────────────
        for w in [60, 300, 600, 1800]:
            hi = sub['close'].rolling(w).max()
            lo = sub['close'].rolling(w).min()
            sub[f'range_{w}'] = hi - lo
        
        sub['range_ratio_s'] = sub['range_60']  / (sub['range_300']  + 1e-9)
        sub['range_ratio_l'] = sub['range_300'] / (sub['range_1800'] + 1e-9)
        
        # ── Autocorrelation ───────────────────────────────────────
        for w in [120, 300, 600]:
            sub[f'autocorr_{w}'] = sub['ret'].rolling(w).apply(
                lambda x: np.corrcoef(x[:-10], x[10:])[0,1]
                if len(x) > 10 else np.nan, raw=True
            )
        
        # ── Depth Imbalance ───────────────────────────────────────
        for lv in range(1, 6):
            bq = sub[f'bid_q{lv}']
            aq = sub[f'ask_q{lv}']
            sub[f'depth_imb_{lv}'] = (bq - aq) / (bq + aq + 1)
        
        sub['depth_imb_total'] = sub[[f'depth_imb_{l}'
                                       for l in range(1,6)]].mean(axis=1)
        
        # ── Price Impact ──────────────────────────────────────────
        sub['price_impact'] = (sub['close'] - sub['mid']).abs()
        sub['pi_z'] = (sub['price_impact'] /
                       (sub['price_impact'].rolling(300).std() + 1e-9))
        
        # ── Forward Returns (labels) ──────────────────────────────
        for h in [10, 30, 60, 120, 300, 600, 900, 1800, 3600]:
            sub[f'fwd_{h}'] = sub['close'].shift(-h) - sub['close']
        
        sub['date'] = d
        feature_dfs.append(sub)
    
    return pd.concat(feature_dfs, ignore_index=True).reset_index(drop=True)

print('Building features...')
df_feat = build_features(df_raw)
print(f'Done. Shape: {df_feat.shape}')
print(f'Columns: {len(df_feat.columns)}')

Building features...
Done. Shape: (85629, 184)
Columns: 184


In [23]:
# ── Layer 1 Extension: Missing Data Sources ────────────────────────
# Build all 5 missing data sources:
# 1. India VIX (NSE, free)
# 2. RBI Reference Rate (RBI, free)  
# 3. DXY (FRED API, free)
# 4. USDINR Options IV (Zerodha, free)
# 5. Calendar features (pure math)

import requests
from datetime import datetime, date, timedelta
import time

# ── Source 1: India VIX ───────────────────────────────────────────
def fetch_india_vix():
    """
    Fetch India VIX from NSE website
    NSE publishes VIX every 15 minutes during market hours
    """
    url = 'https://www.nseindia.com/api/allIndices'
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Accept': 'application/json',
        'Referer': 'https://www.nseindia.com',
    }
    try:
        session = requests.Session()
        # Need to visit NSE first to get cookies
        session.get('https://www.nseindia.com', headers=headers, timeout=10)
        resp = session.get(url, headers=headers, timeout=10)
        data = resp.json()
        for idx in data.get('data', []):
            if idx.get('index') == 'INDIA VIX':
                return {
                    'vix':        float(idx['last']),
                    'vix_chg':    float(idx['change']),
                    'vix_chg_pct':float(idx['percentChange']),
                    'timestamp':  datetime.now().isoformat()
                }
    except Exception as e:
        print(f'VIX fetch error: {e}')
        return None

# Test
vix_data = fetch_india_vix()
print(f'India VIX: {vix_data}')

VIX fetch error: 'change'
India VIX: None


In [24]:
# Debug NSE response
import requests

session = requests.Session()
headers = {
    'User-Agent': 'Mozilla/5.0',
    'Accept':     'application/json',
    'Referer':    'https://www.nseindia.com',
}
session.get('https://www.nseindia.com', headers=headers, timeout=10)
resp = session.get('https://www.nseindia.com/api/allIndices',
                   headers=headers, timeout=10)
data = resp.json()

# Find VIX entry
for idx in data.get('data', []):
    if 'VIX' in str(idx.get('index','')):
        print('VIX entry found:')
        print(idx)
        break
else:
    print('VIX not found. Available indices:')
    for idx in data.get('data', [])[:5]:
        print(f'  {idx.get("index")}: keys={list(idx.keys())}')

VIX entry found:
{'key': 'BROAD MARKET INDICES', 'index': 'INDIA VIX', 'indexSymbol': 'INDIA VIX', 'last': 14.44, 'variation': 0.95, 'percentChange': 7.04, 'open': 13.49, 'high': 15.69, 'low': 13.49, 'previousClose': 13.49, 'yearHigh': 28.91, 'yearLow': 8.72, 'indicativeClose': 0, 'pe': '0', 'pb': '0', 'dy': '0', 'perChange365d': 40.33, 'perChange30d': 24.59, 'date365dAgo': '01-Oct-2025', 'date30dAgo': '02-Sep-2026', 'previousDay': '01-Oct-2026', 'oneWeekAgo': '25-Sep-2026', 'oneMonthAgoVal': 11.59, 'oneWeekAgoVal': 12.16, 'oneYearAgoVal': 10.29, 'previousDayVal': 14.46, 'chart365dPath': 'https://nsearchives.nseindia.com/365d/INDIA-VIX.svg', 'chart30dPath': 'https://nsearchives.nseindia.com/30d/INDIA-VIX.svg', 'chartTodayPath': 'https://nsearchives.nseindia.com/today/INDIA-VIX.svg'}


In [25]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, date, timedelta
import time

# ── Source 1: India VIX ───────────────────────────────────────────
def fetch_india_vix():
    session = requests.Session()
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Accept':     'application/json',
        'Referer':    'https://www.nseindia.com',
    }
    try:
        session.get('https://www.nseindia.com', headers=headers, timeout=10)
        resp = session.get('https://www.nseindia.com/api/allIndices',
                           headers=headers, timeout=10)
        data = resp.json()
        for idx in data.get('data', []):
            if idx.get('index') == 'INDIA VIX':
                return {
                    'vix':          float(idx['last']),
                    'vix_chg':      float(idx['variation']),
                    'vix_chg_pct':  float(idx['percentChange']),
                    'vix_high':     float(idx['high']),
                    'vix_low':      float(idx['low']),
                    'vix_prev':     float(idx['previousDayVal']),
                    'vix_1w_ago':   float(idx['oneWeekAgoVal']),
                    'vix_1m_ago':   float(idx['oneMonthAgoVal']),
                    'timestamp':    datetime.now().isoformat()
                }
    except Exception as e:
        print(f'VIX error: {e}')
        return None

vix = fetch_india_vix()
print(f'✓ India VIX: {vix["vix"]}  chg={vix["vix_chg"]:+.2f} ({vix["vix_chg_pct"]:+.1f}%)')
print(f'  1w ago: {vix["vix_1w_ago"]}  1m ago: {vix["vix_1m_ago"]}')

# ── Source 2: RBI Reference Rate ──────────────────────────────────
def fetch_rbi_rate():
    """
    RBI publishes USDINR reference rate daily at 1:30pm IST
    Available from RBI API
    """
    try:
        # RBI DBIE API
        url = 'https://api.rbi.org.in/api/v1/GetExchangeRate'
        params = {
            'Currency': 'USD',
            'StartDate': (date.today() - timedelta(days=7)).strftime('%d/%m/%Y'),
            'EndDate':   date.today().strftime('%d/%m/%Y')
        }
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        if data and 'ExchangeRate' in data:
            rates = data['ExchangeRate']
            if rates:
                latest = rates[-1]
                return {
                    'rbi_rate':  float(latest.get('USD', 0)),
                    'date':      latest.get('Date',''),
                    'timestamp': datetime.now().isoformat()
                }
    except Exception as e:
        print(f'RBI API error: {e}')
    
    # Fallback: use FRED for USD/INR spot approximation
    try:
        fred_url = 'https://api.stlouisfed.org/fred/series/observations'
        params = {
            'series_id':       'DEXINUS',
            'api_key':         'demo',
            'file_type':       'json',
            'observation_start': (date.today()-timedelta(days=7)).isoformat(),
            'sort_order':      'desc',
            'limit':           1
        }
        resp = requests.get(fred_url, params=params, timeout=10)
        data = resp.json()
        obs  = data.get('observations', [])
        if obs:
            return {
                'rbi_rate':  float(obs[0]['value']),
                'date':      obs[0]['date'],
                'source':    'FRED_DEXINUS',
                'timestamp': datetime.now().isoformat()
            }
    except Exception as e:
        print(f'FRED fallback error: {e}')
    return None

rbi = fetch_rbi_rate()
print(f'\n✓ RBI/Spot Rate: {rbi}')

# ── Source 3: DXY from FRED ───────────────────────────────────────
def fetch_dxy():
    """
    US Dollar Index from FRED API (free, no key needed for basic)
    DTWEXBGS = Trade Weighted US Dollar Index
    """
    try:
        url = 'https://api.stlouisfed.org/fred/series/observations'
        params = {
            'series_id':        'DTWEXBGS',
            'api_key':          'demo',
            'file_type':        'json',
            'observation_start': (date.today()-timedelta(days=30)).isoformat(),
            'sort_order':       'desc',
            'limit':            5
        }
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        obs  = [o for o in data.get('observations',[]) if o['value'] != '.']
        if obs:
            latest = float(obs[0]['value'])
            prev   = float(obs[1]['value']) if len(obs) > 1 else latest
            return {
                'dxy':         latest,
                'dxy_chg':     latest - prev,
                'dxy_chg_pct': (latest - prev) / prev * 100,
                'date':        obs[0]['date'],
                'timestamp':   datetime.now().isoformat()
            }
    except Exception as e:
        print(f'DXY error: {e}')
    return None

dxy = fetch_dxy()
print(f'\n✓ DXY: {dxy}')

# ── Source 4: USDINR Options IV ───────────────────────────────────
def fetch_usdinr_options_iv(kite, futures_price, expiry_date):
    """
    Fetch ATM options IV from Zerodha
    kite:          KiteConnect instance
    futures_price: current USDINR futures price
    expiry_date:   contract expiry date
    """
    try:
        # Round to nearest 0.25 for strike
        atm_strike = round(futures_price * 4) / 4
        strikes    = [atm_strike - 0.25, atm_strike, atm_strike + 0.25]
        
        expiry_str = expiry_date.strftime('%y%b').upper()
        instruments = []
        for s in strikes:
            strike_int = int(s * 100)
            instruments += [
                f'CDS:USDINR{expiry_str}{strike_int}CE',
                f'CDS:USDINR{expiry_str}{strike_int}PE',
            ]
        
        quotes = kite.quote(instruments)
        
        results = {}
        for inst, q in quotes.items():
            results[inst] = {
                'last_price': q.get('last_price', 0),
                'oi':         q.get('oi', 0),
                'volume':     q.get('volume', 0),
            }
        
        return results
    except Exception as e:
        print(f'Options error: {e}')
        return None

# ── Source 5: Calendar Features (pure math) ───────────────────────
def build_calendar_features(df):
    """
    Add calendar-based features to dataframe
    These are deterministic — no data fetching needed
    """
    df = df.copy()
    
    # RBI MPC meeting dates 2026 (known in advance)
    rbi_dates = [
        date(2026, 2, 7),
        date(2026, 4, 9),
        date(2026, 6, 6),
        date(2026, 8, 8),
        date(2026, 10, 8),  # ← next one (Oct 8 2026)
        date(2026, 12, 5),
    ]
    
    # US FOMC dates 2026 (known in advance)
    fomc_dates = [
        date(2026, 1, 29),
        date(2026, 3, 19),
        date(2026, 5, 7),
        date(2026, 6, 18),
        date(2026, 7, 30),
        date(2026, 9, 17),
        date(2026, 11, 5),
        date(2026, 12, 16),
    ]
    
    # Convert date column
    df['date_obj'] = pd.to_datetime(df['date']).dt.date
    
    # Days to next RBI meeting
    df['days_to_rbi'] = df['date_obj'].apply(
        lambda d: min([(r - d).days for r in rbi_dates if r >= d],
                      default=999)
    )
    
    # Days to next FOMC
    df['days_to_fomc'] = df['date_obj'].apply(
        lambda d: min([(f - d).days for f in fomc_dates if f >= d],
                      default=999)
    )
    
    # Days to contract expiry (OCTFUT expires Oct 28)
    expiry = date(2026, 10, 28)
    df['days_to_expiry'] = df['date_obj'].apply(
        lambda d: max((expiry - d).days, 0)
    )
    
    # Time features
    df['minutes_since_open'] = (df['ist_sec'] - 9*3600) / 60
    df['minutes_to_close']   = (17*3600 - df['ist_sec']) / 60
    df['hour_of_day']        = df['ist_sec'] // 3600
    df['minute_of_hour']     = (df['ist_sec'] % 3600) // 60
    
    # Day of week (0=Mon, 4=Fri)
    df['day_of_week'] = pd.to_datetime(df['date']).dt.dayofweek
    
    # Is near RBI (within 2 days)
    df['near_rbi']  = (df['days_to_rbi']  <= 2).astype(int)
    df['near_fomc'] = (df['days_to_fomc'] <= 2).astype(int)
    
    # Is near expiry (within 5 days)
    df['near_expiry'] = (df['days_to_expiry'] <= 5).astype(int)
    
    # Session (morning/afternoon)
    df['is_morning']   = (df['ist_sec'] < 12*3600).astype(int)
    df['is_afternoon'] = (df['ist_sec'] >= 13*3600).astype(int)
    
    df.drop(columns=['date_obj'], inplace=True)
    return df

# Test calendar features
df_cal = build_calendar_features(df_feat.head(100))
cal_cols = ['minutes_since_open','days_to_rbi','days_to_fomc',
            'days_to_expiry','near_rbi','near_expiry']
print(f'\n✓ Calendar features:')
print(df_cal[cal_cols].iloc[0].to_dict())

print('\n=== Data Source Summary ===')
print(f'India VIX:      {"✓" if vix else "✗"}  {vix["vix"] if vix else "failed"}')
print(f'RBI/Spot Rate:  {"✓" if rbi else "✗"}  {rbi}')
print(f'DXY:            {"✓" if dxy else "✗"}  {dxy["dxy"] if dxy else "failed"}')
print(f'Calendar:       ✓  (pure math, always works)')
print(f'Options IV:     ⚠  needs market hours + valid token')

✓ India VIX: 14.44  chg=+0.95 (+7.0%)
  1w ago: 12.16  1m ago: 11.59
RBI API error: HTTPSConnectionPool(host='api.rbi.org.in', port=443): Max retries exceeded with url: /api/v1/GetExchangeRate?Currency=USD&StartDate=26%2F09%2F2026&EndDate=03%2F10%2F2026 (Caused by NameResolutionError("HTTPSConnection(host='api.rbi.org.in', port=443): Failed to resolve 'api.rbi.org.in' ([Errno 11001] getaddrinfo failed)"))

✓ RBI/Spot Rate: None

✓ DXY: None

✓ Calendar features:
{'minutes_since_open': 26.55, 'days_to_rbi': 16.0, 'days_to_fomc': 44.0, 'days_to_expiry': 36.0, 'near_rbi': 0.0, 'near_expiry': 0.0}

=== Data Source Summary ===
India VIX:      ✓  14.44
RBI/Spot Rate:  ✗  None
DXY:            ✗  failed
Calendar:       ✓  (pure math, always works)
Options IV:     ⚠  needs market hours + valid token


In [26]:
# ── Fix Source 2: RBI Rate (use FBIL rate instead) ─────────────────
def fetch_rbi_rate_v2():
    """
    FBIL (Financial Benchmarks India) publishes USDINR rate
    Accessible via NSE or direct scraping
    """
    try:
        # Try FBIL via NSE API
        session = requests.Session()
        headers = {
            'User-Agent': 'Mozilla/5.0',
            'Accept':     'application/json',
            'Referer':    'https://www.nseindia.com',
        }
        session.get('https://www.nseindia.com', headers=headers, timeout=10)
        
        # NSE CDS market watch gives spot rate
        resp = session.get(
            'https://www.nseindia.com/api/marketStatus',
            headers=headers, timeout=10
        )
        data = resp.json()
        print(f'NSE market status keys: {list(data.keys())}')
        return data
    except Exception as e:
        print(f'NSE error: {e}')
    return None

# Try alternative: use exchangerate-api.com (free)
def fetch_spot_rate():
    """
    Get USDINR spot from free exchange rate API
    """
    try:
        resp = requests.get(
            'https://api.exchangerate-api.com/v4/latest/USD',
            timeout=10
        )
        data = resp.json()
        inr  = data['rates']['INR']
        return {
            'spot_rate': inr,
            'date':      data['date'],
            'timestamp': datetime.now().isoformat(),
            'source':    'exchangerate-api.com'
        }
    except Exception as e:
        print(f'Spot rate error: {e}')
    return None

# ── Fix Source 3: DXY using alternative ───────────────────────────
def fetch_dxy_v2():
    """
    DXY from Yahoo Finance (free, no API key)
    """
    try:
        # Yahoo Finance DXY
        url = 'https://query1.finance.yahoo.com/v8/finance/chart/DX-Y.NYB'
        headers = {'User-Agent': 'Mozilla/5.0'}
        params  = {
            'interval':  '1d',
            'range':     '5d',
        }
        resp = requests.get(url, headers=headers,
                            params=params, timeout=10)
        data = resp.json()
        
        result = data['chart']['result'][0]
        closes = result['indicators']['quote'][0]['close']
        times  = result['timestamp']
        
        # Get last 2 valid closes
        valid = [(t, c) for t, c in zip(times, closes) if c is not None]
        if len(valid) >= 2:
            latest = valid[-1][1]
            prev   = valid[-2][1]
            return {
                'dxy':         round(latest, 2),
                'dxy_prev':    round(prev, 2),
                'dxy_chg':     round(latest - prev, 3),
                'dxy_chg_pct': round((latest - prev)/prev*100, 3),
                'timestamp':   datetime.now().isoformat()
            }
    except Exception as e:
        print(f'Yahoo DXY error: {e}')
    
    # Fallback: use EUR/USD as DXY proxy (inverse)
    try:
        resp = requests.get(
            'https://api.exchangerate-api.com/v4/latest/USD',
            timeout=10
        )
        data  = resp.json()
        eurusd = data['rates']['EUR']
        # DXY proxy: when USD strengthens → EUR/USD falls → DXY rises
        return {
            'dxy_proxy':    round(1/eurusd * 100, 2),
            'eurusd':       round(eurusd, 4),
            'timestamp':    datetime.now().isoformat(),
            'source':       'eurusd_proxy'
        }
    except Exception as e:
        print(f'DXY proxy error: {e}')
    return None

# Test fixed versions
spot = fetch_spot_rate()
dxy2 = fetch_dxy_v2()

print(f'Spot USDINR: {spot}')
print(f'DXY:         {dxy2}')

Spot USDINR: {'spot_rate': 96.37, 'date': '2026-10-03', 'timestamp': '2026-10-03T13:41:03.729501', 'source': 'exchangerate-api.com'}
DXY:         {'dxy': 101.93, 'dxy_prev': 102.1, 'dxy_chg': -0.17, 'dxy_chg_pct': -0.167, 'timestamp': '2026-10-03T13:41:03.815575'}


In [27]:
# ── Complete macro data snapshot ───────────────────────────────────
def fetch_macro_snapshot():
    """
    Fetch all external data in one call
    Returns dict with all macro features
    Run this once at market open and every 15 minutes
    """
    snapshot = {
        'timestamp': datetime.now().isoformat(),
        # Defaults (used if fetch fails)
        'vix':           15.0,
        'vix_chg_pct':   0.0,
        'vix_1w_chg':    0.0,
        'vix_regime':    'NORMAL',  # LOW/NORMAL/HIGH/EXTREME
        'spot_rate':     96.0,
        'futures_basis': 0.0,
        'dxy':           100.0,
        'dxy_chg_pct':   0.0,
        'dxy_regime':    'NEUTRAL', # STRONG/NEUTRAL/WEAK
    }
    
    # VIX
    vix = fetch_india_vix()
    if vix:
        snapshot['vix']         = vix['vix']
        snapshot['vix_chg_pct'] = vix['vix_chg_pct']
        snapshot['vix_1w_chg']  = ((vix['vix'] - vix['vix_1w_ago']) /
                                    (vix['vix_1w_ago'] + 1e-9) * 100)
        # VIX regime
        if vix['vix'] < 12:
            snapshot['vix_regime'] = 'LOW'
        elif vix['vix'] < 18:
            snapshot['vix_regime'] = 'NORMAL'
        elif vix['vix'] < 25:
            snapshot['vix_regime'] = 'HIGH'
        else:
            snapshot['vix_regime'] = 'EXTREME'
    
    # Spot rate
    spot = fetch_spot_rate()
    if spot:
        snapshot['spot_rate'] = spot['spot_rate']
    
    # DXY
    dxy = fetch_dxy_v2()
    if dxy:
        snapshot['dxy']         = dxy.get('dxy', dxy.get('dxy_proxy', 100))
        snapshot['dxy_chg_pct'] = dxy.get('dxy_chg_pct', 0)
        if dxy.get('dxy_chg_pct', 0) > 0.3:
            snapshot['dxy_regime'] = 'STRONG'
        elif dxy.get('dxy_chg_pct', 0) < -0.3:
            snapshot['dxy_regime'] = 'WEAK'
        else:
            snapshot['dxy_regime'] = 'NEUTRAL'
    
    return snapshot

# Test
macro = fetch_macro_snapshot()
print('=== Macro Snapshot ===')
for k, v in macro.items():
    print(f'  {k:<20}: {v}')

=== Macro Snapshot ===
  timestamp           : 2026-10-03T13:41:39.557517
  vix                 : 14.44
  vix_chg_pct         : 7.04
  vix_1w_chg          : 18.74999999845805
  vix_regime          : NORMAL
  spot_rate           : 96.37
  futures_basis       : 0.0
  dxy                 : 101.93
  dxy_chg_pct         : -0.167
  dxy_regime          : NEUTRAL


In [29]:
# ── Add macro features to df_feat ─────────────────────────────────
def add_macro_features(df, macro_snapshot):
    df = df.copy()
    
    # VIX features
    df['vix']         = macro_snapshot['vix']
    df['vix_chg_pct'] = macro_snapshot['vix_chg_pct']
    df['vix_1w_chg']  = macro_snapshot['vix_1w_chg']
    df['vix_low']     = int(macro_snapshot['vix'] < 12)
    df['vix_normal']  = int(12 <= macro_snapshot['vix'] < 18)
    df['vix_high']    = int(macro_snapshot['vix'] >= 18)
    
    # DXY features
    df['dxy']         = macro_snapshot['dxy']
    df['dxy_chg_pct'] = macro_snapshot['dxy_chg_pct']
    df['dxy_strong']  = int(macro_snapshot['dxy_regime'] == 'STRONG')
    df['dxy_weak']    = int(macro_snapshot['dxy_regime'] == 'WEAK')
    
    # Spot rate features
    df['spot_rate']     = macro_snapshot['spot_rate']
    df['futures_basis'] = df['close'] - macro_snapshot['spot_rate']
    df['basis_z']       = (df['futures_basis'] /
                           (df['futures_basis'].rolling(300).std() + 1e-9))
    
    return df

# Add calendar features
df_feat = build_calendar_features(df_feat)

# Add macro (today's snapshot for all days — approximation for backtest)
df_feat = add_macro_features(df_feat, macro)

# ── Define complete feature set ────────────────────────────────────
FEATURE_COLS = {
    
    # Group 1: Realized Volatility (13 features)
    'volatility': [
        'rv_10','rv_30','rv_60','rv_300','rv_1800',
        'rv_10_30','rv_30_60','rv_60_300','rv_300_1800',
        'rv_accel','rv_trend','rv_trend2',
    ],
    
    # Group 2: Spread (9 features)
    'spread': [
        'spread_bps',
        'spread_z_30','spread_z_300',
        'spread_trend_s','spread_trend_l','spread_accel',
    ],
    
    # Group 3: Order Flow (10 features)
    'ofi': [
        'ofi_norm','ofi_1_norm',
        'ofi_ma_10','ofi_ma_30','ofi_ma_300',
        'ofi_trend_s','ofi_trend_l','ofi_accel',
    ],
    
    # Group 4: Cancel Imbalance (12 features) ← PRIMARY SIGNAL
    'cancel': [
        'cancel_ma_30','cancel_ma_300',
        'cancel_z_30','cancel_z_60','cancel_z_300','cancel_z_600',
        'cancel_trend_s','cancel_trend_l','cancel_accel',
    ],
    
    # Group 5: Weighted Mid (7 features)
    'wmid': [
        'wmid_z_30','wmid_z_60','wmid_z_300',
        'wmid_trend','wmid_accel',
    ],
    
    # Group 6: Momentum (11 features)
    'momentum': [
        'mom_10','mom_30','mom_60','mom_300','mom_1800',
        'mom_s_m','mom_m_l','mom_l_vl',
    ],
    
    # Group 7: VWAP (3 features)
    'vwap': [
        'vwap_dev','vwap_z_300','vwap_z_1800',
    ],
    
    # Group 8: Volume (6 features)
    'volume': [
        'vol_ratio_30','vol_ratio_300',
        'vol_accel','vol_trend',
    ],
    
    # Group 9: OI (6 features)
    'oi': [
        'oi_z_30','oi_z_300',
        'oi_trend_s','oi_trend_l',
    ],
    
    # Group 10: Range (6 features)
    'range': [
        'range_60','range_300','range_1800',
        'range_ratio_s','range_ratio_l',
    ],
    
    # Group 11: Autocorrelation (3 features)
    'autocorr': [
        'autocorr_120','autocorr_300','autocorr_600',
    ],
    
    # Group 12: Depth (6 features)
    'depth': [
        'depth_imb_1','depth_imb_2','depth_imb_3',
        'depth_imb_4','depth_imb_5','depth_imb_total',
    ],
    
    # Group 13: Price Impact (1 feature)
    'price_impact': ['pi_z'],
    
    # Group 14: Macro (8 features)
    'macro': [
        'vix','vix_chg_pct','vix_1w_chg',
        'vix_high',
        'dxy','dxy_chg_pct',
        'futures_basis','basis_z',
    ],
    
    # Group 15: Calendar (9 features)
    'calendar': [
        'minutes_since_open','minutes_to_close',
        'days_to_rbi','days_to_fomc','days_to_expiry',
        'near_rbi','near_fomc','near_expiry',
        'day_of_week',
    ],
}

# Flatten all features
ALL_FEATURES = [f for grp in FEATURE_COLS.values() for f in grp]

# Check which features exist
missing = [f for f in ALL_FEATURES if f not in df_feat.columns]
present = [f for f in ALL_FEATURES if f in df_feat.columns]

print(f'Total features defined:  {len(ALL_FEATURES)}')
print(f'Present in dataframe:    {len(present)}')
print(f'Missing:                 {len(missing)}')
if missing:
    print(f'Missing cols: {missing}')

# Drop rows with NaN in core features
core_features = FEATURE_COLS['cancel'] + FEATURE_COLS['momentum']
fwd_cols      = [f'fwd_{h}' for h in [60, 300, 1800]]

df_clean = df_feat.dropna(
    subset=core_features + ['fwd_1800']
).reset_index(drop=True)

print(f'\nClean bars: {len(df_clean):,}')
print(f'Days:       {df_clean["date"].nunique()}')
print()
print('=== Feature Groups ===')
for grp, feats in FEATURE_COLS.items():
    avail = [f for f in feats if f in df_clean.columns]
    print(f'  {grp:<15}: {len(avail)}/{len(feats)} features')

Total features defined:  91
Present in dataframe:    91
Missing:                 0

Clean bars: 61,108
Days:       6

=== Feature Groups ===
  volatility     : 12/12 features
  spread         : 6/6 features
  ofi            : 8/8 features
  cancel         : 9/9 features
  wmid           : 5/5 features
  momentum       : 8/8 features
  vwap           : 3/3 features
  volume         : 4/4 features
  oi             : 4/4 features
  range          : 5/5 features
  autocorr       : 3/3 features
  depth          : 6/6 features
  price_impact   : 1/1 features
  macro          : 8/8 features
  calendar       : 9/9 features


In [30]:
# ── Layer 3: Hamilton HMM Regime Detection ─────────────────────────
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

# Regime features — capture market microstructure state
# NOT signal features — these describe WHEN to trade
REGIME_FEATURES = [
    # Volatility structure
    'rv_60_300', 'rv_1800', 'rv_trend',
    # Market maker activity
    'spread_bps', 'spread_trend_s',
    # Order flow
    'ofi_norm', 'depth_imb_total',
    # Price structure
    'range_ratio_l', 'autocorr_300',
    # Activity level
    'vol_ratio_300',
    # Fair value deviation
    'vwap_dev',
]

df_hmm = df_clean.dropna(subset=REGIME_FEATURES).reset_index(drop=True)

# Fit per day lengths — critical for Hamilton model
lengths = []
for d in sorted(df_hmm['date'].unique()):
    n = (df_hmm['date'] == d).sum()
    lengths.append(n)
    print(f'{d}: {n} bars')

print(f'\nTotal: {sum(lengths):,} bars')

# Standardize
scaler_hmm = StandardScaler()
X_hmm      = scaler_hmm.fit_transform(df_hmm[REGIME_FEATURES].values)

# Fit 4-state Hamilton HMM
print('\nFitting Hamilton HMM...')
hmm = GaussianHMM(
    n_components    = 4,
    covariance_type = 'full',
    n_iter          = 200,
    random_state    = 42,
    tol             = 1e-4
)
hmm.fit(X_hmm, lengths)

states = hmm.predict(X_hmm, lengths)
probs  = hmm.predict_proba(X_hmm, lengths)

df_hmm['state']  = states
for i in range(4):
    df_hmm[f'p_s{i}'] = probs[:, i]

print(f'Log likelihood: {hmm.score(X_hmm, lengths):.2f}')
print()

# State distribution
print('=== State Distribution ===')
for s in range(4):
    print(f'  State {s}: {(states==s).mean()*100:.1f}%')

print()
print('=== Feature Means by State ===')
print(f'{"Feature":<20} {"St0":>8} {"St1":>8} {"St2":>8} {"St3":>8}')
print('-'*52)
for f in REGIME_FEATURES:
    v = [df_hmm[df_hmm['state']==s][f].mean() for s in range(4)]
    print(f'{f:<20} {v[0]:>8.4f} {v[1]:>8.4f} {v[2]:>8.4f} {v[3]:>8.4f}')

print()
print('=== Transition Matrix ===')
print('From\\To    St0     St1     St2     St3')
for i in range(4):
    r = hmm.transmat_[i]
    print(f'State {i}   {r[0]:.3f}   {r[1]:.3f}   {r[2]:.3f}   {r[3]:.3f}')

2026-09-22: 8968 bars
2026-09-23: 7431 bars
2026-09-24: 6785 bars
2026-09-25: 11808 bars
2026-09-30: 13067 bars
2026-10-01: 12519 bars

Total: 60,578 bars

Fitting Hamilton HMM...
Log likelihood: -673793.70

=== State Distribution ===
  State 0: 15.5%
  State 1: 9.9%
  State 2: 58.0%
  State 3: 16.5%

=== Feature Means by State ===
Feature                   St0      St1      St2      St3
----------------------------------------------------
rv_60_300              0.4664   0.3941   0.3666   0.3141
rv_1800                0.0009   0.0009   0.0007   0.0013
rv_trend              -0.0002  -0.0002  -0.0002  -0.0003
spread_bps             1.1555   0.7751   0.7992   1.0643
spread_trend_s         0.0016  -0.0007  -0.0003   0.0002
ofi_norm               0.1037   0.0127  -0.0008   0.0235
depth_imb_total        0.2378   0.1494   0.0453   0.2274
range_ratio_l          0.4561   0.3893   0.3627   0.4364
autocorr_300          -0.0021  -0.0150  -0.0160  -0.0087
vol_ratio_300          1.0046   1.7764   0.

In [31]:
# ── Label states and check IC ──────────────────────────────────────
STATE_LABELS = {
    0: 'BID_PRESSURE',   # ofi=0.104, depth=0.238, vwap=-4.76 (below fair value)
    1: 'HIGH_ACTIVITY',  # vol_ratio=1.776, unstable (24% sticky) → AVOID
    2: 'BASE_QUIET',     # spread=0.799, balanced, 90.6% sticky → PRIMARY
    3: 'ASK_PRESSURE',   # vwap=+9.82, wide spread, 86.3% sticky
}

# Merge state back into df_clean
df_clean = df_clean.merge(
    df_hmm[['date','ist_sec','state','p_s0','p_s1','p_s2','p_s3']],
    on=['date','ist_sec'],
    how='left'
)

print('=== IC of cancel_z_300 per State ===')
print(f'{"State":<8} {"Label":<15}', end='')
for h in [60, 300, 900, 1800, 3600]:
    print(f'{h}s'.rjust(8), end='')
print(f'  {"n":>8}  {"pct":>6}')
print('-'*70)

for s in range(4):
    mask = df_clean['state'] == s
    print(f'State {s}  {STATE_LABELS[s]:<15}', end='')
    for h in [60, 300, 900, 1800, 3600]:
        col = f'fwd_{h}'
        v   = df_clean[mask].dropna(subset=['cancel_z_300', col])
        ic  = v['cancel_z_300'].corr(v[col]) if len(v) > 100 else np.nan
        print(f'{ic:>8.3f}', end='')
    n   = mask.sum()
    pct = mask.mean()*100
    print(f'  {n:>8}  {pct:>5.1f}%')

# Best trading states
print()
print('=== Trading Decision ===')
print('State 0 (BID_PRESSURE):  cancel_z_300 IC=-0.406 → TRADE SHORT')
print('State 1 (HIGH_ACTIVITY): IC=-0.256, unstable    → AVOID')
print('State 2 (BASE_QUIET):    cancel_z_300 IC=-0.287 → TRADE SHORT')
print('State 3 (ASK_PRESSURE):  cancel_z_300 IC=-0.268 → TRADE SHORT')
print()
print('TRADE in States: 0, 2, 3 (90.1% of time)')
print('AVOID State 1 (9.9% of time)')

=== IC of cancel_z_300 per State ===
State    Label               60s    300s    900s   1800s   3600s         n     pct
----------------------------------------------------------------------
State 0  BID_PRESSURE     -0.143  -0.092  -0.194  -0.406  -0.472      9419   15.4%
State 1  HIGH_ACTIVITY    -0.016  -0.093  -0.203  -0.256  -0.331      5986    9.8%
State 2  BASE_QUIET       -0.027  -0.125  -0.207  -0.287  -0.284     35159   57.5%
State 3  ASK_PRESSURE     -0.083  -0.224  -0.330  -0.268  -0.425     10014   16.4%

=== Trading Decision ===
State 0 (BID_PRESSURE):  cancel_z_300 IC=-0.406 → TRADE SHORT
State 1 (HIGH_ACTIVITY): IC=-0.256, unstable    → AVOID
State 2 (BASE_QUIET):    cancel_z_300 IC=-0.287 → TRADE SHORT
State 3 (ASK_PRESSURE):  cancel_z_300 IC=-0.268 → TRADE SHORT

TRADE in States: 0, 2, 3 (90.1% of time)
AVOID State 1 (9.9% of time)


In [32]:
# ── Layer 4: Primary Signal Model ─────────────────────────────────
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Signal features — predictive features, not regime features
SIGNAL_FEATURES = [
    # Cancel imbalance (primary)
    'cancel_z_300', 'cancel_z_30', 'cancel_ma_300',
    # Depth
    'depth_imb_total', 'depth_imb_1',
    # Momentum
    'mom_60', 'mom_1800', 'mom_l_vl',
    # Volatility
    'rv_1800', 'rv_300_1800', 'rv_trend',
    # Spread
    'spread_bps', 'spread_trend_s',
    # VWAP
    'vwap_dev', 'vwap_z_300',
    # Autocorrelation
    'autocorr_300', 'autocorr_600',
    # OFI
    'ofi_norm', 'ofi_ma_30', 'ofi_trend_s',
    # Weighted mid
    'wmid_z_300', 'wmid_trend',
    # Macro
    'vix', 'vix_chg_pct', 'dxy_chg_pct', 'futures_basis',
    # Calendar
    'minutes_since_open', 'days_to_rbi', 'days_to_expiry',
    # HMM state probabilities
    'p_s0', 'p_s1', 'p_s2', 'p_s3',
]

# Check all present
missing_sig = [f for f in SIGNAL_FEATURES if f not in df_clean.columns]
print(f'Signal features: {len(SIGNAL_FEATURES)}')
print(f'Missing: {missing_sig}')

# ── Walk-Forward Validation ────────────────────────────────────────
TARGET   = 'fwd_1800'
dates    = sorted(df_clean['date'].unique())
print(f'\nDates: {dates}')
print()

wf_results = []

print('=== Walk-Forward Ridge Regression (1800s target) ===')
print(f'{"Train days":<12} {"Test date":<14} {"OOS IC":>8} '
      f'{"n_test":>8} {"alpha":>8}')
print('-'*52)

for test_idx in range(2, len(dates)):
    train_dates = dates[:test_idx]
    test_date   = dates[test_idx]
    
    train = df_clean[
        df_clean['date'].isin(train_dates)
    ].dropna(subset=SIGNAL_FEATURES+[TARGET])
    
    test = df_clean[
        df_clean['date']==test_date
    ].dropna(subset=SIGNAL_FEATURES+[TARGET])
    
    if len(train) < 500 or len(test) < 100:
        continue
    
    sc      = StandardScaler()
    X_train = sc.fit_transform(train[SIGNAL_FEATURES].values)
    X_test  = sc.transform(test[SIGNAL_FEATURES].values)
    y_train = train[TARGET].values
    y_test  = test[TARGET].values
    
    model   = RidgeCV(alphas=[0.1, 1, 10, 100, 1000])
    model.fit(X_train, y_train)
    
    preds  = model.predict(X_test)
    ic_oos = pd.Series(preds).corr(pd.Series(y_test))
    
    wf_results.append({
        'train_days': len(train_dates),
        'test_date':  test_date,
        'ic_oos':     ic_oos,
        'n_test':     len(test),
        'alpha':      model.alpha_,
        'model':      model,
        'scaler':     sc,
    })
    
    print(f'{len(train_dates):<12} {test_date:<14} {ic_oos:>8.4f} '
          f'{len(test):>8} {model.alpha_:>8.1f}')

df_wf = pd.DataFrame([{k:v for k,v in r.items()
                        if k not in ['model','scaler']}
                       for r in wf_results])

print()
print(f'Avg OOS IC:     {df_wf["ic_oos"].mean():.4f}')
print(f'Positive days:  {(df_wf["ic_oos"]>0).mean()*100:.0f}%')
print(f'Min OOS IC:     {df_wf["ic_oos"].min():.4f}')
print(f'Max OOS IC:     {df_wf["ic_oos"].max():.4f}')

Signal features: 33
Missing: []

Dates: ['2026-09-22', '2026-09-23', '2026-09-24', '2026-09-25', '2026-09-30', '2026-10-01']

=== Walk-Forward Ridge Regression (1800s target) ===
Train days   Test date        OOS IC   n_test    alpha
----------------------------------------------------
2            2026-09-24       0.1507     6785      0.1
3            2026-09-25       0.0262    11808      1.0
4            2026-09-30       0.2719    13067     10.0
5            2026-10-01       0.4131    12519      1.0

Avg OOS IC:     0.2155
Positive days:  100%
Min OOS IC:     0.0262
Max OOS IC:     0.4131


In [33]:
# ── Layer 5: Meta-Labeling ─────────────────────────────────────────
# Step 1: Primary model predicts direction
# Step 2: Meta model predicts IF primary model is correct
# Only trade when meta model says primary is reliable

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import precision_score

print('=== Layer 5: Meta-Labeling ===')
print()

# Use last model (trained on 5 days, tested on Oct 1)
last_result = wf_results[-1]
model_last  = last_result['model']
scaler_last  = last_result['scaler']

# Get test data
test_date  = '2026-10-01'
train_dates = dates[:-1]

train = df_clean[
    df_clean['date'].isin(train_dates)
].dropna(subset=SIGNAL_FEATURES+[TARGET])

test = df_clean[
    df_clean['date']==test_date
].dropna(subset=SIGNAL_FEATURES+[TARGET])

# Primary model predictions on training data
X_train  = scaler_last.transform(train[SIGNAL_FEATURES].values)
X_test   = scaler_last.transform(test[SIGNAL_FEATURES].values)

train_preds = model_last.predict(X_train)
test_preds  = model_last.predict(X_test)

# ── Triple Barrier Labeling ────────────────────────────────────────
def triple_barrier(prices, tp=0.0010, sl=0.0008, max_bars=1800):
    """
    Label each bar:
      +1: price hits TP first (good trade)
      -1: price hits SL first (bad trade)
       0: time barrier (inconclusive)
    """
    n      = len(prices)
    labels = np.zeros(n)
    for i in range(n):
        entry = prices[i]
        for j in range(1, min(max_bars+1, n-i)):
            p = prices[i+j]
            if p >= entry*(1+tp):
                labels[i] =  1
                break
            elif p <= entry*(1-sl):
                labels[i] = -1
                break
    return labels

# Apply triple barrier to training data
print('Building triple barrier labels...')
tb_labels = {}
for d in sorted(train['date'].unique()):
    day    = train[train['date']==d].copy().reset_index(drop=True)
    prices = day['close'].values
    labels = triple_barrier(prices, tp=0.0010, sl=0.0008, max_bars=1800)
    for i, idx in enumerate(day.index):
        tb_labels[idx] = labels[i]

train['tb_label'] = train.index.map(tb_labels)
train_tb = train[train['tb_label'] != 0].copy()
print(f'Labeled bars: {len(train_tb):,}')
print(f'  TP hit: {(train_tb["tb_label"]==1).sum()} ({(train_tb["tb_label"]==1).mean()*100:.1f}%)')
print(f'  SL hit: {(train_tb["tb_label"]==-1).sum()} ({(train_tb["tb_label"]==-1).mean()*100:.1f}%)')

# ── Meta features ──────────────────────────────────────────────────
# What does the meta model need to know?
# - Primary model prediction strength
# - Current regime
# - Signal confidence

META_FEATURES = SIGNAL_FEATURES + [
    'state',  # HMM regime
]

# Add primary model prediction as meta feature
train_tb['primary_pred'] = model_last.predict(
    scaler_last.transform(train_tb[SIGNAL_FEATURES].values)
)
train_tb['pred_strength'] = train_tb['primary_pred'].abs()

meta_features_final = META_FEATURES + ['primary_pred', 'pred_strength']
meta_features_final = [f for f in meta_features_final
                       if f in train_tb.columns]

# ── Train meta model ───────────────────────────────────────────────
# Binary: is primary model correct? (1=yes, 0=no)
train_tb['meta_target'] = (
    (train_tb['primary_pred'] > 0) == (train_tb['tb_label'] == 1)
).astype(int)

print(f'\nMeta target distribution:')
print(f'  Primary correct: {train_tb["meta_target"].mean()*100:.1f}%')

X_meta = train_tb[meta_features_final].fillna(0).values
y_meta = train_tb['meta_target'].values

sc_meta   = StandardScaler()
X_meta_sc = sc_meta.fit_transform(X_meta)

meta_model = LogisticRegressionCV(
    cv=3, max_iter=500, random_state=42
)
meta_model.fit(X_meta_sc, y_meta)

is_acc = (meta_model.predict(X_meta_sc) == y_meta).mean()
print(f'Meta model IS accuracy: {is_acc:.3f}')

# ── Apply meta model to test ───────────────────────────────────────
test['primary_pred'] = test_preds
test['pred_strength']= test['primary_pred'].abs()
meta_feat_test       = [f for f in meta_features_final if f in test.columns]

X_test_meta  = sc_meta.transform(test[meta_feat_test].fillna(0).values)
meta_probs   = meta_model.predict_proba(X_test_meta)[:, 1]
test['meta_prob'] = meta_probs

# Compare: primary alone vs primary + meta filter
print()
print('=== Meta-Labeling Effect on Test Day ===')
for meta_thr in [0.5, 0.55, 0.60, 0.65, 0.70]:
    # With meta filter
    meta_mask = test['meta_prob'] > meta_thr
    if meta_mask.sum() < 50:
        continue
    
    sub    = test[meta_mask]
    ic_filtered = sub['primary_pred'].corr(sub[TARGET])
    
    print(f'Meta threshold {meta_thr:.2f}: '
          f'n={meta_mask.sum():>5}  '
          f'IC={ic_filtered:.4f}  '
          f'({meta_mask.mean()*100:.1f}% of bars)')

print()
# Without meta filter
ic_full = pd.Series(test_preds).corr(pd.Series(test[TARGET].values))
print(f'Without meta filter: IC={ic_full:.4f}  n={len(test)}')

=== Layer 5: Meta-Labeling ===

Building triple barrier labels...
Labeled bars: 38,128
  TP hit: 0 (0.0%)
  SL hit: 3136 (8.2%)

Meta target distribution:
  Primary correct: 65.3%
Meta model IS accuracy: 0.999

=== Meta-Labeling Effect on Test Day ===
Meta threshold 0.50: n=10102  IC=0.7982  (80.7% of bars)
Meta threshold 0.55: n=10101  IC=0.7985  (80.7% of bars)
Meta threshold 0.60: n=10099  IC=0.7984  (80.7% of bars)
Meta threshold 0.65: n=10098  IC=0.7983  (80.7% of bars)
Meta threshold 0.70: n=10097  IC=0.7982  (80.7% of bars)

Without meta filter: IC=0.4131  n=12519


In [34]:
# ── Fix Triple Barrier ─────────────────────────────────────────────
# USDINR tick = 0.0025 paise
# Spread = ~0.008 paise
# Need TP > spread to be meaningful

# Check actual price moves in our data
print('=== Price Move Analysis ===')
for d in sorted(train['date'].unique()):
    day = train[train['date']==d].copy()
    moves_30  = day['close'].diff(30).abs().dropna()
    moves_300 = day['close'].diff(300).abs().dropna()
    moves_1800= day['close'].diff(1800).abs().dropna()
    print(f'{d}: avg_30s={moves_30.mean():.4f}  '
          f'avg_300s={moves_300.mean():.4f}  '
          f'avg_1800s={moves_1800.mean():.4f}  '
          f'p90_1800s={moves_1800.quantile(0.9):.4f}')

print()
print('=== Calibrating TP/SL ===')
all_moves = []
for d in sorted(train['date'].unique()):
    day = train[train['date']==d].copy()
    all_moves.append(day['close'].diff(1800).abs().dropna())
moves = pd.concat(all_moves)
print(f'1800s move percentiles:')
for p in [10, 25, 50, 75, 90]:
    print(f'  p{p}: {moves.quantile(p/100):.4f} paise')

=== Price Move Analysis ===
2026-09-22: avg_30s=0.0064  avg_300s=0.0202  avg_1800s=0.0689  p90_1800s=0.1600
2026-09-23: avg_30s=0.0050  avg_300s=0.0150  avg_1800s=0.0307  p90_1800s=0.0675
2026-09-24: avg_30s=0.0061  avg_300s=0.0116  avg_1800s=0.0309  p90_1800s=0.0700
2026-09-25: avg_30s=0.0040  avg_300s=0.0135  avg_1800s=0.0313  p90_1800s=0.0650
2026-09-30: avg_30s=0.0044  avg_300s=0.0143  avg_1800s=0.0373  p90_1800s=0.0800

=== Calibrating TP/SL ===
1800s move percentiles:
  p10: 0.0050 paise
  p25: 0.0125 paise
  p50: 0.0275 paise
  p75: 0.0550 paise
  p90: 0.0850 paise


In [35]:
# ── Recalibrate Triple Barrier ─────────────────────────────────────
# p50 move at 1800s = 0.0275 paise
# TP = p50 = 0.0275 paise (achievable ~50% of time)
# SL = p25 = 0.0125 paise (conservative stop)

# In absolute paise terms (not percentage)
TP_PAISE = 0.0300  # 30% above median move
SL_PAISE = 0.0125  # 25th percentile move

def triple_barrier_abs(prices, tp=0.0300, sl=0.0125, max_bars=1800):
    """
    Triple barrier with ABSOLUTE paise thresholds
    NOT percentage — correct for USDINR
    """
    n      = len(prices)
    labels = np.zeros(n)
    for i in range(n):
        entry = prices[i]
        tp_level = entry + tp  # SHORT: we want price to fall
        sl_level = entry - sl  # SHORT: stop if price rises
        # For SHORT signal:
        #   TP = entry - tp (price falls by tp)
        #   SL = entry + sl (price rises by sl)
        for j in range(1, min(max_bars+1, n-i)):
            p = prices[i+j]
            if p <= entry - tp:   # TP hit (price fell)
                labels[i] =  1
                break
            elif p >= entry + sl: # SL hit (price rose)
                labels[i] = -1
                break
    return labels

# Apply to all training days
print('Building calibrated triple barrier labels...')
tb_labels_v2 = {}
for d in sorted(train['date'].unique()):
    day    = train[train['date']==d].copy().reset_index(drop=True)
    prices = day['close'].values
    labels = triple_barrier_abs(prices, tp=TP_PAISE, sl=SL_PAISE)
    for i, idx in enumerate(day.index):
        tb_labels_v2[idx] = labels[i]

train['tb_label'] = train.index.map(tb_labels_v2)
train_tb = train[train['tb_label'] != 0].copy()

print(f'Total labeled:    {len(train_tb):,}')
print(f'TP hit (+1):      {(train_tb["tb_label"]==1).sum():,} ({(train_tb["tb_label"]==1).mean()*100:.1f}%)')
print(f'SL hit (-1):      {(train_tb["tb_label"]==-1).sum():,} ({(train_tb["tb_label"]==-1).mean()*100:.1f}%)')
print(f'Inconclusive (0): {(train["tb_label"]==0).sum():,} ({(train["tb_label"]==0).mean()*100:.1f}%)')

print()
# Check IC of cancel_z_300 vs triple barrier label
# Negative cancel_z_300 → price should fall → label = +1 (TP hit)
labeled = train_tb.dropna(subset=['cancel_z_300'])
ic_tb   = labeled['cancel_z_300'].corr(labeled['tb_label'])
print(f'cancel_z_300 IC vs triple barrier: {ic_tb:.4f}')
print(f'(Negative = when cancel_z_300 high → SL hits more)')
print(f'(Positive = when cancel_z_300 high → TP hits more)')

Building calibrated triple barrier labels...
Total labeled:    48,009
TP hit (+1):      6,278 (13.1%)
SL hit (-1):      6,739 (14.0%)
Inconclusive (0): 50 (0.1%)

cancel_z_300 IC vs triple barrier: 0.0481
(Negative = when cancel_z_300 high → SL hits more)
(Positive = when cancel_z_300 high → TP hits more)


In [36]:
# ── Verify signal direction vs labels ─────────────────────────────
print('=== Signal Direction Check ===')
print()

# When cancel_z_300 is NEGATIVE (our SHORT signal):
# → We expect price to FALL
# → For SHORT: price fall = TP hit = label +1
# → So: cancel_z_300 < 0 should correlate with label = +1

neg_cancel = train_tb[train_tb['cancel_z_300'] < -0.5]
pos_cancel = train_tb[train_tb['cancel_z_300'] >  0.5]

print(f'When cancel_z_300 < -0.5 (SHORT signal):')
print(f'  TP hit: {(neg_cancel["tb_label"]==1).mean()*100:.1f}%')
print(f'  SL hit: {(neg_cancel["tb_label"]==-1).mean()*100:.1f}%')
print(f'  n = {len(neg_cancel)}')
print()
print(f'When cancel_z_300 > +0.5 (no signal):')
print(f'  TP hit: {(pos_cancel["tb_label"]==1).mean()*100:.1f}%')
print(f'  SL hit: {(pos_cancel["tb_label"]==-1).mean()*100:.1f}%')
print(f'  n = {len(pos_cancel)}')

print()
# Check IC at different thresholds
print('=== cancel_z_300 IC vs label by threshold ===')
print(f'{"Threshold":>10} {"n_short":>8} {"TP%":>8} {"SL%":>8}')
print('-'*36)
for thr in [0.3, 0.5, 0.7, 1.0, 1.5, 2.0]:
    mask = train_tb['cancel_z_300'] < -thr
    sub  = train_tb[mask]
    if len(sub) < 50: continue
    tp_pct = (sub['tb_label']==1).mean()*100
    sl_pct = (sub['tb_label']==-1).mean()*100
    print(f'{-thr:>10.1f} {len(sub):>8} {tp_pct:>7.1f}% {sl_pct:>7.1f}%')

print()
# Now build proper meta model
# Meta target: did the SHORT trade hit TP?
# Only consider bars where cancel_z_300 < -0.3 (signal fires)
signal_bars = train_tb[train_tb['cancel_z_300'] < -0.3].copy()
signal_bars['meta_target'] = (signal_bars['tb_label'] == 1).astype(int)

print(f'Signal bars (cancel_z_300 < -0.3): {len(signal_bars):,}')
print(f'TP rate on signal bars: {signal_bars["meta_target"].mean()*100:.1f}%')
print()
print('=== Meta features importance check ===')
# Quick IC check of meta features vs meta_target
meta_check_features = [
    'state', 'p_s0', 'p_s1', 'p_s2', 'p_s3',
    'vix', 'vix_chg_pct',
    'dxy_chg_pct', 'futures_basis',
    'minutes_since_open', 'days_to_rbi',
    'rv_1800', 'spread_bps',
    'vol_ratio_300', 'autocorr_300',
]
print(f'{"Feature":<25} {"IC_vs_meta_target":>18}')
print('-'*45)
for f in meta_check_features:
    if f not in signal_bars.columns: continue
    ic = signal_bars[f].corr(signal_bars['meta_target'])
    if abs(ic) > 0.02:
        print(f'{f:<25} {ic:>18.4f}')

=== Signal Direction Check ===

When cancel_z_300 < -0.5 (SHORT signal):
  TP hit: nan%
  SL hit: nan%
  n = 0

When cancel_z_300 > +0.5 (no signal):
  TP hit: 0.0%
  SL hit: 1.2%
  n = 4829

=== cancel_z_300 IC vs label by threshold ===
 Threshold  n_short      TP%      SL%
------------------------------------
      -0.3      994     0.0%    12.5%

Signal bars (cancel_z_300 < -0.3): 994
TP rate on signal bars: 0.0%

=== Meta features importance check ===
Feature                    IC_vs_meta_target
---------------------------------------------


In [37]:
# ── Debug triple barrier ───────────────────────────────────────────
print('=== Debug ===')

# Check one day manually
d   = '2026-09-22'
day = df_clean[df_clean['date']==d].copy().reset_index(drop=True)

# Find a bar where cancel_z_300 < -0.5
signal_bars_day = day[day['cancel_z_300'] < -0.5]
print(f'Sep 22 bars with cancel_z_300 < -0.5: {len(signal_bars_day)}')

if len(signal_bars_day) > 0:
    # Take first signal bar
    idx   = signal_bars_day.index[0]
    entry = day.loc[idx, 'close']
    ist   = day.loc[idx, 'ist_sec']
    czval = day.loc[idx, 'cancel_z_300']
    
    print(f'  Bar idx={idx}, ist={ist}, close={entry:.4f}')
    print(f'  cancel_z_300={czval:.4f}')
    
    # Check next 1800 bars
    future = day.loc[idx:idx+1800, 'close'].values
    tp_level = entry - TP_PAISE  # SHORT: TP when price falls
    sl_level = entry + SL_PAISE  # SHORT: SL when price rises
    
    print(f'  TP level: {tp_level:.4f} (need price to fall to here)')
    print(f'  SL level: {sl_level:.4f} (stop if price rises here)')
    print(f'  Next 10 closes: {future[:10]}')
    print(f'  Min price in 1800s: {future.min():.4f}')
    print(f'  Max price in 1800s: {future.max():.4f}')
    print(f'  TP hit? {future.min() <= tp_level}')
    print(f'  SL hit? {future.max() >= sl_level}')

# Also check actual cancel_z_300 distribution
print()
print(f'cancel_z_300 stats in training data:')
print(df_clean[df_clean['date'].isin(dates[:-1])]['cancel_z_300'].describe())

=== Debug ===
Sep 22 bars with cancel_z_300 < -0.5: 0

cancel_z_300 stats in training data:
count    48154.000000
mean         0.129884
std          0.271458
min         -0.494969
25%         -0.109441
50%          0.108917
75%          0.391436
max          0.643089
Name: cancel_z_300, dtype: float64


In [38]:
print('=== cancel_z_300 distribution ===')
print(f'Min:  {df_clean["cancel_z_300"].min():.4f}')
print(f'Max:  {df_clean["cancel_z_300"].max():.4f}')
print(f'Mean: {df_clean["cancel_z_300"].mean():.4f}')
print(f'Std:  {df_clean["cancel_z_300"].std():.4f}')
print()
print('Percentiles:')
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f'  p{p:>2}: {df_clean["cancel_z_300"].quantile(p/100):.4f}')

print()
# Check how cancel_z_300 was built
# In build_features: cancel_z_w = cancel_ma_w / (cancel_imb.rolling(w).std() + 1e-9)
# This is NOT clipped — but values are small
# The IC analysis uses this raw value

# Recheck IC with actual values
v = df_clean.dropna(subset=['cancel_z_300','fwd_1800'])
print(f'IC cancel_z_300 vs fwd_1800: {v["cancel_z_300"].corr(v["fwd_1800"]):.4f}')
print(f'Bars with cancel_z_300 < -0.3: {(df_clean["cancel_z_300"]<-0.3).sum()}')
print(f'Bars with cancel_z_300 < -0.1: {(df_clean["cancel_z_300"]<-0.1).sum()}')

=== cancel_z_300 distribution ===
Min:  -0.6122
Max:  0.6444
Mean: 0.0972
Std:  0.2723

Percentiles:
  p 1: -0.4420
  p 5: -0.2750
  p10: -0.2032
  p25: -0.1184
  p50: 0.0311
  p75: 0.3558
  p90: 0.4918
  p95: 0.5395
  p99: 0.5866

IC cancel_z_300 vs fwd_1800: -0.3140
Bars with cancel_z_300 < -0.3: 2071
Bars with cancel_z_300 < -0.1: 18396


In [39]:
# ── Fix: use correct thresholds ───────────────────────────────────
# Signal threshold: p10 = -0.20 (bottom 10% of cancel_z_300)
# TP/SL: based on actual price moves

SIGNAL_THR = -0.20   # p10 of cancel_z_300
TP_PAISE   = 0.0275  # p50 of 1800s move
SL_PAISE   = 0.0125  # p25 of 1800s move

print(f'Signal threshold: {SIGNAL_THR} (p10 of cancel_z_300)')
print(f'TP: {TP_PAISE} paise  SL: {SL_PAISE} paise')
print()

# Rebuild triple barrier per day on df_clean
print('Building triple barrier labels (all days)...')
all_tb = []
for d in sorted(df_clean['date'].unique()):
    day    = df_clean[df_clean['date']==d].copy().reset_index(drop=True)
    prices = day['close'].values
    labels = triple_barrier_abs(prices, tp=TP_PAISE,
                                sl=SL_PAISE, max_bars=1800)
    day['tb_label'] = labels
    all_tb.append(day)

df_tb = pd.concat(all_tb, ignore_index=True)

# Stats
labeled = df_tb[df_tb['tb_label'] != 0]
print(f'Total bars:       {len(df_tb):,}')
print(f'Labeled bars:     {len(labeled):,} ({len(labeled)/len(df_tb)*100:.1f}%)')
print(f'TP hit (+1):      {(labeled["tb_label"]==1).sum():,} ({(labeled["tb_label"]==1).mean()*100:.1f}%)')
print(f'SL hit (-1):      {(labeled["tb_label"]==-1).sum():,} ({(labeled["tb_label"]==-1).mean()*100:.1f}%)')

# Signal bars
signal_bars = df_tb[
    (df_tb['cancel_z_300'] < SIGNAL_THR) &
    (df_tb['tb_label'] != 0)
].copy()

print()
print(f'Signal bars (cancel_z_300 < {SIGNAL_THR}): {len(signal_bars):,}')
print(f'TP rate on signal bars: {(signal_bars["tb_label"]==1).mean()*100:.1f}%')
print(f'SL rate on signal bars: {(signal_bars["tb_label"]==-1).mean()*100:.1f}%')

print()
# IC of cancel_z_300 vs triple barrier label
ic_tb = signal_bars['cancel_z_300'].corr(signal_bars['tb_label'])
print(f'cancel_z_300 IC vs triple barrier: {ic_tb:.4f}')

# By state
print()
print('=== TP rate per state ===')
print(f'{"State":<8} {"Label":<15} {"TP%":>6} {"SL%":>6} {"n":>6}')
print('-'*42)
for s in range(4):
    mask = (signal_bars['state'] == s)
    sub  = signal_bars[mask]
    if len(sub) < 10: continue
    tp = (sub['tb_label']==1).mean()*100
    sl = (sub['tb_label']==-1).mean()*100
    print(f'State {s}  {STATE_LABELS[s]:<15} {tp:>5.1f}% {sl:>5.1f}% {len(sub):>6}')

Signal threshold: -0.2 (p10 of cancel_z_300)
TP: 0.0275 paise  SL: 0.0125 paise

Building triple barrier labels (all days)...
Total bars:       61,108
Labeled bars:     59,377 (97.2%)
TP hit (+1):      19,162 (32.3%)
SL hit (-1):      40,215 (67.7%)

Signal bars (cancel_z_300 < -0.2): 5,988
TP rate on signal bars: 27.5%
SL rate on signal bars: 72.5%

cancel_z_300 IC vs triple barrier: 0.1653

=== TP rate per state ===
State    Label              TP%    SL%      n
------------------------------------------
State 0  BID_PRESSURE     55.1%  44.9%    671
State 1  HIGH_ACTIVITY    31.6%  68.4%    567
State 2  BASE_QUIET       25.4%  74.6%   3393
State 3  ASK_PRESSURE     25.2%  74.8%    922


In [40]:
# ── Meta model: predict when TP will hit ──────────────────────────
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# Train on days 1-5, test on day 6
train_dates = dates[:-1]
test_date   = dates[-1]

train_tb = df_tb[df_tb['date'].isin(train_dates)].copy()
test_tb  = df_tb[df_tb['date']==test_date].copy()

# Only use signal bars
train_signal = train_tb[
    (train_tb['cancel_z_300'] < SIGNAL_THR) &
    (train_tb['tb_label'] != 0)
].copy()
test_signal = test_tb[
    (test_tb['cancel_z_300'] < SIGNAL_THR) &
    (test_tb['tb_label'] != 0)
].copy()

train_signal['meta_target'] = (train_signal['tb_label']==1).astype(int)
test_signal['meta_target']  = (test_signal['tb_label']==1).astype(int)

print(f'Train signal bars: {len(train_signal):,}')
print(f'Test signal bars:  {len(test_signal):,}')
print(f'Train TP rate: {train_signal["meta_target"].mean()*100:.1f}%')
print(f'Test TP rate:  {test_signal["meta_target"].mean()*100:.1f}%')
print()

# Meta features — what predicts WHEN signal works
META_FEATURES = [
    # Regime
    'state', 'p_s0', 'p_s1', 'p_s2', 'p_s3',
    # Signal strength
    'cancel_z_300', 'cancel_z_30', 'cancel_ma_300',
    # Market conditions
    'spread_bps', 'rv_1800', 'vol_ratio_300',
    'autocorr_300', 'vwap_dev',
    # Depth
    'depth_imb_total',
    # Macro
    'vix', 'vix_chg_pct', 'dxy_chg_pct',
    'futures_basis',
    # Calendar
    'minutes_since_open', 'days_to_rbi',
    'days_to_expiry',
]

META_FEATURES = [f for f in META_FEATURES
                 if f in train_signal.columns]

X_train_m = train_signal[META_FEATURES].fillna(0).values
y_train_m = train_signal['meta_target'].values
X_test_m  = test_signal[META_FEATURES].fillna(0).values
y_test_m  = test_signal['meta_target'].values

sc_meta   = StandardScaler()
X_train_m = sc_meta.fit_transform(X_train_m)
X_test_m  = sc_meta.transform(X_test_m)

meta_model = LogisticRegressionCV(
    cv=3, max_iter=500,
    class_weight='balanced',
    random_state=42
)
meta_model.fit(X_train_m, y_train_m)

# Evaluate
train_probs = meta_model.predict_proba(X_train_m)[:,1]
test_probs  = meta_model.predict_proba(X_test_m)[:,1]

train_auc = roc_auc_score(y_train_m, train_probs)
test_auc  = roc_auc_score(y_test_m,  test_probs)

print(f'Meta model AUC:')
print(f'  Train: {train_auc:.4f}')
print(f'  Test:  {test_auc:.4f}')
print()

# Meta threshold analysis
print('=== Meta Threshold Analysis (test day) ===')
print(f'{"Threshold":>10} {"n":>6} {"TP%":>8} {"Coverage":>10}')
print('-'*38)
for thr in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    mask = test_probs > thr
    if mask.sum() < 5: continue
    tp   = y_test_m[mask].mean()*100
    cov  = mask.mean()*100
    print(f'{thr:>10.2f} {mask.sum():>6} {tp:>7.1f}% {cov:>9.1f}%')

print()
print(f'Baseline (no meta filter):')
print(f'  n={len(test_signal)} TP={y_test_m.mean()*100:.1f}%')

# Feature importance
coef = pd.Series(meta_model.coef_[0], index=META_FEATURES)
print()
print('=== Top Meta Features ===')
print(coef.abs().sort_values(ascending=False).head(10).to_string())

Train signal bars: 3,899
Test signal bars:  2,089
Train TP rate: 29.1%
Test TP rate:  24.4%

Meta model AUC:
  Train: 0.8354
  Test:  0.5464

=== Meta Threshold Analysis (test day) ===
 Threshold      n      TP%   Coverage
--------------------------------------
      0.30   1369    20.8%      65.5%
      0.35   1223    22.6%      58.5%
      0.40   1054    26.3%      50.5%
      0.45    959    28.8%      45.9%
      0.50    814    31.3%      39.0%
      0.55    733    33.7%      35.1%
      0.60    644    37.1%      30.8%

Baseline (no meta filter):
  n=2089 TP=24.4%

=== Top Meta Features ===
vwap_dev              0.843398
rv_1800               0.659306
p_s0                  0.388307
state                 0.340676
autocorr_300          0.323843
cancel_ma_300         0.323390
futures_basis         0.275992
minutes_since_open    0.275059
p_s3                  0.186275
depth_imb_total       0.146937


In [41]:
# ── Layer 6: Kelly Position Sizing + Complete Backtest ─────────────

def kelly_lots(p_win, avg_win_paise, avg_loss_paise,
               max_lots=50, min_lots=5, fraction=0.25):
    """
    Kelly Criterion for position sizing
    fraction=0.25 = quarter Kelly (conservative)
    """
    if avg_loss_paise <= 0 or p_win <= 0:
        return min_lots
    b     = avg_win_paise / avg_loss_paise
    q     = 1 - p_win
    kelly = (p_win * b - q) / b
    kelly = max(0, kelly)
    lots  = int(kelly * fraction * max_lots)
    return max(min_lots, min(lots, max_lots))

# Calibrate from training data
train_tp_bars = train_signal[train_signal['tb_label']==1]
train_sl_bars = train_signal[train_signal['tb_label']==-1]

AVG_WIN_PAISE  = TP_PAISE   # 0.0275
AVG_LOSS_PAISE = SL_PAISE   # 0.0125
P_WIN_BASE     = train_signal['meta_target'].mean()

print('=== Kelly Position Sizing ===')
print(f'Avg win:  {AVG_WIN_PAISE:.4f} paise = Rs.{AVG_WIN_PAISE*1000*20:.0f} (20 lots)')
print(f'Avg loss: {AVG_LOSS_PAISE:.4f} paise = Rs.{AVG_LOSS_PAISE*1000*20:.0f} (20 lots)')
print()
print(f'{"P(win)":>8} {"Kelly_lots":>12} {"Expected_PnL":>14}')
print('-'*36)
for p in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]:
    lots   = kelly_lots(p, AVG_WIN_PAISE, AVG_LOSS_PAISE)
    ev     = (p*AVG_WIN_PAISE - (1-p)*AVG_LOSS_PAISE) * lots * 1000
    print(f'{p:>8.2f} {lots:>12} Rs.{ev:>10.0f}')

print()

# ── Complete Walk-Forward Backtest ─────────────────────────────────
print('=== Complete Architecture Backtest ===')
print()

LOT_SIZE    = 1000
MAX_LOTS    = 50
MIN_LOTS    = 5
META_THR    = 0.55   # 33.7% TP rate
HOLD_MAX    = 1800
EOD_IST     = 17*3600 - 120

all_trades = []

# Walk-forward: train on all previous days, test on next
for test_idx in range(2, len(dates)):
    train_dates_wf = dates[:test_idx]
    test_date_wf   = dates[test_idx]
    
    # Training data
    train_wf = df_tb[df_tb['date'].isin(train_dates_wf)].copy()
    train_sig_wf = train_wf[
        (train_wf['cancel_z_300'] < SIGNAL_THR) &
        (train_wf['tb_label'] != 0)
    ].copy()
    train_sig_wf['meta_target'] = (train_sig_wf['tb_label']==1).astype(int)
    
    if len(train_sig_wf) < 100:
        continue
    
    # Train meta model
    mf_avail = [f for f in META_FEATURES if f in train_sig_wf.columns]
    X_tr = train_sig_wf[mf_avail].fillna(0).values
    y_tr = train_sig_wf['meta_target'].values
    sc   = StandardScaler()
    X_tr = sc.fit_transform(X_tr)
    mm   = LogisticRegressionCV(cv=3, max_iter=500,
                                 class_weight='balanced',
                                 random_state=42)
    mm.fit(X_tr, y_tr)
    
    # Calibrate Kelly
    p_win_cal = train_sig_wf['meta_target'].mean()
    
    # Test day
    test_day = df_tb[df_tb['date']==test_date_wf].copy().reset_index(drop=True)
    
    inv = 0; ep = 0.0; et = 0; current_lots = MIN_LOTS
    
    for i, row in test_day.iterrows():
        ist    = int(row['ist_sec'])
        czval  = float(row['cancel_z_300']) if not pd.isna(row['cancel_z_300']) else 0
        bid_p1 = float(row['bid_p1'])
        ask_p1 = float(row['ask_p1'])
        
        # EOD close
        if ist >= EOD_IST and inv != 0:
            xp  = ask_p1
            pnl = (ep - xp) * current_lots * LOT_SIZE - 65
            all_trades.append({
                'date':   test_date_wf,
                'ist':    ist,
                'pnl':    pnl,
                'lots':   current_lots,
                'reason': 'EOD',
                'hold':   ist - et,
                'state':  int(row['state']) if not pd.isna(row['state']) else -1,
            })
            inv = 0
            break
        
        # Exit on time stop
        if inv == -1 and (ist - et) >= HOLD_MAX:
            xp  = ask_p1
            pnl = (ep - xp) * current_lots * LOT_SIZE - 65
            all_trades.append({
                'date':   test_date_wf,
                'ist':    ist,
                'pnl':    pnl,
                'lots':   current_lots,
                'reason': 'TIME',
                'hold':   ist - et,
                'state':  int(row['state']) if not pd.isna(row['state']) else -1,
            })
            inv = 0
        
        # Entry
        if inv == 0 and czval < SIGNAL_THR:
            # Compute meta probability
            mf_row = pd.DataFrame([row[mf_avail].fillna(0).values],
                                   columns=mf_avail)
            X_row    = sc.transform(mf_row.values)
            meta_p   = mm.predict_proba(X_row)[0, 1]
            
            if meta_p > META_THR:
                # Kelly sizing based on meta probability
                lots = kelly_lots(meta_p, AVG_WIN_PAISE,
                                  AVG_LOSS_PAISE)
                ep            = bid_p1
                et            = ist
                inv           = -1
                current_lots  = lots

# ── Results ────────────────────────────────────────────────────────
if all_trades:
    df_trades = pd.DataFrame(all_trades)
    pnls = df_trades['pnl']
    wins = pnls[pnls > 0]

    print(f'Total trades:   {len(df_trades)}')
    print(f'Win rate:       {len(wins)/len(pnls)*100:.1f}%')
    print(f'Total PnL:      Rs.{pnls.sum():,.0f}')
    print(f'Avg PnL:        Rs.{pnls.mean():,.0f}')
    print(f'Max win:        Rs.{pnls.max():,.0f}')
    print(f'Max loss:       Rs.{pnls.min():,.0f}')
    print(f'Avg hold:       {df_trades["hold"].mean():.0f}s')
    print(f'Avg lots:       {df_trades["lots"].mean():.1f}')
    print()
    print('Per day:')
    for d, g in df_trades.groupby('date'):
        wr = (g['pnl']>0).mean()*100
        print(f'  {d}: {len(g):>3} trades  '
              f'win={wr:.0f}%  '
              f'lots={g["lots"].mean():.1f}  '
              f'Rs.{g["pnl"].sum():>8,.0f}')
    print()
    print('By state:')
    for s in range(4):
        g = df_trades[df_trades['state']==s]
        if len(g) == 0: continue
        wr = (g['pnl']>0).mean()*100
        print(f'  State {s} ({STATE_LABELS[s]:<15}): '
              f'{len(g):>3} trades  win={wr:.0f}%  '
              f'Rs.{g["pnl"].sum():>8,.0f}')

=== Kelly Position Sizing ===
Avg win:  0.0275 paise = Rs.550 (20 lots)
Avg loss: 0.0125 paise = Rs.250 (20 lots)

  P(win)   Kelly_lots   Expected_PnL
------------------------------------
    0.25            5 Rs.       -13
    0.30            5 Rs.        -2
    0.35            5 Rs.         7
    0.40            5 Rs.        18
    0.45            5 Rs.        27
    0.50            5 Rs.        38
    0.55            5 Rs.        48

=== Complete Architecture Backtest ===

Total trades:   6
Win rate:       0.0%
Total PnL:      Rs.-1,975
Avg PnL:        Rs.-329
Max win:        Rs.-65
Max loss:       Rs.-765
Avg hold:       1816s
Avg lots:       8.7

Per day:
  2026-09-24:   2 trades  win=0%  lots=12.0  Rs.    -910
  2026-10-01:   4 trades  win=0%  lots=7.0  Rs.  -1,065

By state:
  State 1 (HIGH_ACTIVITY  ):   1 trades  win=0%  Rs.    -120
  State 2 (BASE_QUIET     ):   1 trades  win=0%  Rs.    -185
  State 3 (ASK_PRESSURE   ):   4 trades  win=0%  Rs.  -1,670


In [42]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ── Step 1: Build correlation matrix ──────────────────────────────
print('=== Step 1: Correlation Matrix ===')

# Use our signal features
SIGNAL_FEATURES = [
    'cancel_z_300', 'cancel_z_30', 'cancel_ma_300',
    'depth_imb_total', 'depth_imb_1',
    'mom_60', 'mom_1800', 'mom_l_vl',
    'rv_1800', 'rv_300_1800', 'rv_trend',
    'spread_bps', 'spread_trend_s',
    'vwap_dev', 'vwap_z_300',
    'autocorr_300', 'autocorr_600',
    'ofi_norm', 'ofi_ma_30', 'ofi_trend_s',
    'wmid_z_300', 'wmid_trend',
    'vix', 'vix_chg_pct', 'dxy_chg_pct',
    'futures_basis',
    'minutes_since_open', 'days_to_rbi',
    'days_to_expiry',
]

# Get clean data
df_sig = df_clean.dropna(subset=SIGNAL_FEATURES).reset_index(drop=True)

N = len(SIGNAL_FEATURES)   # features
T = len(df_sig)             # observations
q = T / N                   # ratio

print(f'Features (N): {N}')
print(f'Observations (T): {T:,}')
print(f'q = T/N = {q:.1f}')
print()

# Standardize
sc      = StandardScaler()
X       = sc.fit_transform(df_sig[SIGNAL_FEATURES].values)

# Correlation matrix
corr    = np.corrcoef(X.T)
print(f'Correlation matrix shape: {corr.shape}')
print(f'Mean abs correlation: {np.abs(corr - np.eye(N)).mean():.4f}')
print(f'Max abs correlation:  {np.abs(corr - np.eye(N)).max():.4f}')

# ── Step 2: Eigenvalue decomposition ──────────────────────────────
print()
print('=== Step 2: Eigenvalue Decomposition ===')

eigenvalues, eigenvectors = np.linalg.eigh(corr)

# Sort descending
idx         = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors= eigenvectors[:, idx]

print(f'Top 10 eigenvalues:')
for i, ev in enumerate(eigenvalues[:10]):
    print(f'  λ{i+1}: {ev:.4f}')

print()
print(f'Sum of eigenvalues: {eigenvalues.sum():.2f} (should = {N})')
print(f'Min eigenvalue: {eigenvalues.min():.4f}')
print(f'Max eigenvalue: {eigenvalues.max():.4f}')

Task was destroyed but it is pending!
task: <Task pending name='Task-539' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:76> wait_for=<Task pending name='Task-540' coro=<_async_in_context.<locals>.preserve_context() running at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\projects\hedge_fund\venv\lib\site-packages\zmq\eventloop\zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-540' coro=<_async_in_context.<locals>.preserve_context() running at C:\projects\hedge_fund\venv\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]>


=== Step 1: Correlation Matrix ===
Features (N): 29
Observations (T): 60,578
q = T/N = 2088.9

Correlation matrix shape: (29, 29)
Mean abs correlation: nan
Max abs correlation:  nan

=== Step 2: Eigenvalue Decomposition ===
Top 10 eigenvalues:
  λ1: nan
  λ2: nan
  λ3: nan
  λ4: nan
  λ5: nan
  λ6: nan
  λ7: nan
  λ8: nan
  λ9: nan
  λ10: nan

Sum of eigenvalues: nan (should = 29)
Min eigenvalue: nan
Max eigenvalue: nan


In [43]:
# ── Debug NaN issue ────────────────────────────────────────────────
print('=== NaN Check ===')
for f in SIGNAL_FEATURES:
    nan_pct = df_sig[f].isna().mean()*100
    inf_pct = np.isinf(df_sig[f].replace([np.inf,-np.inf], np.nan).fillna(0)).mean()*100
    if nan_pct > 0 or inf_pct > 0:
        print(f'{f:<25}: NaN={nan_pct:.1f}%  Inf={inf_pct:.1f}%')

# Fix: replace inf with NaN then fill
print()
print('Fixing...')
df_sig2 = df_sig[SIGNAL_FEATURES].copy()
df_sig2 = df_sig2.replace([np.inf, -np.inf], np.nan)

# Fill NaN with rolling median per feature
for f in SIGNAL_FEATURES:
    if df_sig2[f].isna().any():
        df_sig2[f] = df_sig2[f].fillna(df_sig2[f].median())

# Check again
still_bad = [f for f in SIGNAL_FEATURES
             if df_sig2[f].isna().any() or
             np.isinf(df_sig2[f]).any()]
print(f'Still bad after fix: {still_bad}')

# Standardize and compute correlation
sc   = StandardScaler()
X    = sc.fit_transform(df_sig2.values)

corr = np.corrcoef(X.T)
print(f'NaN in corr: {np.isnan(corr).sum()}')
print(f'Mean abs corr: {np.abs(corr - np.eye(len(SIGNAL_FEATURES))).mean():.4f}')

# Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eigh(corr)
idx          = eigenvalues.argsort()[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print()
print('Top 10 eigenvalues:')
for i, ev in enumerate(eigenvalues[:10]):
    print(f'  λ{i+1}: {ev:.4f}')
print(f'Sum: {eigenvalues.sum():.2f} (should = {len(SIGNAL_FEATURES)})')

=== NaN Check ===

Fixing...
Still bad after fix: []
NaN in corr: 165
Mean abs corr: nan

Top 10 eigenvalues:
  λ1: nan
  λ2: nan
  λ3: nan
  λ4: nan
  λ5: nan
  λ6: nan
  λ7: nan
  λ8: nan
  λ9: nan
  λ10: nan
Sum: nan (should = 29)


In [44]:
# ── Deep fix ──────────────────────────────────────────────────────
print('=== Feature Variance Check ===')

df_sig2 = df_sig[SIGNAL_FEATURES].copy()
df_sig2 = df_sig2.replace([np.inf, -np.inf], np.nan)
df_sig2 = df_sig2.fillna(df_sig2.median())

# Check variance
for f in SIGNAL_FEATURES:
    std = df_sig2[f].std()
    if std < 1e-10:
        print(f'{f}: ZERO VARIANCE = {std}')
    elif df_sig2[f].isna().any():
        print(f'{f}: still has NaN')

# Remove zero-variance features
valid_features = [f for f in SIGNAL_FEATURES
                  if df_sig2[f].std() > 1e-10
                  and not df_sig2[f].isna().any()]

print(f'\nValid features: {len(valid_features)}/{len(SIGNAL_FEATURES)}')
removed = set(SIGNAL_FEATURES) - set(valid_features)
if removed:
    print(f'Removed: {removed}')

# Rebuild with valid features only
X    = df_sig2[valid_features].values.astype(np.float64)

# Manual correlation (more robust than np.corrcoef)
X_std = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-9)
corr  = X_std.T @ X_std / len(X_std)

print(f'\nCorrelation matrix shape: {corr.shape}')
print(f'NaN in corr: {np.isnan(corr).sum()}')
print(f'Mean abs corr: {np.abs(corr - np.eye(len(valid_features))).mean():.4f}')

# Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eigh(corr)
idx          = eigenvalues.argsort()[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

N = len(valid_features)
T = len(X)
q = T / N

print(f'\nq = T/N = {q:.1f}')
print(f'\nTop 10 eigenvalues:')
for i, ev in enumerate(eigenvalues[:10]):
    pct = ev/eigenvalues.sum()*100
    print(f'  λ{i+1}: {ev:.4f}  ({pct:.1f}% variance)')
print(f'Sum: {eigenvalues.sum():.2f} (should = {N})')

=== Feature Variance Check ===
vix: ZERO VARIANCE = 0.0
vix_chg_pct: ZERO VARIANCE = 3.552743002628166e-15
dxy_chg_pct: ZERO VARIANCE = 5.55116094160651e-17

Valid features: 26/29
Removed: {'vix_chg_pct', 'dxy_chg_pct', 'vix'}

Correlation matrix shape: (26, 26)
NaN in corr: 0
Mean abs corr: 0.1465

q = T/N = 2329.9

Top 10 eigenvalues:
  λ1: 5.3537  (20.6% variance)
  λ2: 3.5619  (13.7% variance)
  λ3: 2.5489  (9.8% variance)
  λ4: 1.8587  (7.1% variance)
  λ5: 1.6768  (6.4% variance)
  λ6: 1.6031  (6.2% variance)
  λ7: 1.4172  (5.5% variance)
  λ8: 1.3306  (5.1% variance)
  λ9: 1.0439  (4.0% variance)
  λ10: 0.9719  (3.7% variance)
Sum: 26.00 (should = 26)


In [45]:
# ── Step 3: Marchenko-Pastur Distribution ─────────────────────────
from scipy.optimize import minimize
from scipy.stats import chi2

print('=== Step 3: Marchenko-Pastur Fitting ===')
print()

def marchenko_pastur_pdf(x, q, sigma=1.0):
    """
    Marchenko-Pastur probability density function
    
    q    = T/N (observations/features ratio)
    sigma= variance of random matrix entries
    
    λ_min = σ²(1 - 1/√q)²
    λ_max = σ²(1 + 1/√q)²
    """
    lambda_min = sigma**2 * (1 - 1/np.sqrt(q))**2
    lambda_max = sigma**2 * (1 + 1/np.sqrt(q))**2
    
    # PDF is zero outside [lambda_min, lambda_max]
    pdf = np.zeros_like(x, dtype=float)
    mask = (x >= lambda_min) & (x <= lambda_max)
    
    if mask.any():
        pdf[mask] = (q / (2 * np.pi * sigma**2 * x[mask])) * \
                    np.sqrt((lambda_max - x[mask]) *
                            (x[mask] - lambda_min))
    return pdf

def fit_mp_sigma(eigenvalues, q):
    """
    Fit sigma² by minimizing SSE between
    empirical eigenvalue distribution and MP PDF
    """
    def sse(sigma):
        # Theoretical PDF
        x       = np.linspace(eigenvalues.min(),
                              eigenvalues.max(), 1000)
        pdf_mp  = marchenko_pastur_pdf(x, q, sigma[0])
        
        # Empirical PDF (histogram)
        hist, edges = np.histogram(eigenvalues, bins=50,
                                   density=True)
        x_emp = (edges[:-1] + edges[1:]) / 2
        pdf_th = marchenko_pastur_pdf(x_emp, q, sigma[0])
        
        return np.sum((hist - pdf_th)**2)
    
    result = minimize(sse, x0=[1.0],
                      bounds=[(0.01, 2.0)],
                      method='L-BFGS-B')
    return result.x[0]

# Fit sigma
sigma_fit = fit_mp_sigma(eigenvalues, q)
print(f'Fitted sigma²: {sigma_fit:.4f}')

# Compute MP bounds
lambda_min = sigma_fit**2 * (1 - 1/np.sqrt(q))**2
lambda_max = sigma_fit**2 * (1 + 1/np.sqrt(q))**2

print(f'MP λ_min: {lambda_min:.4f}')
print(f'MP λ_max: {lambda_max:.4f}')
print()

# Classify eigenvalues
signal_mask = eigenvalues > lambda_max
noise_mask  = eigenvalues <= lambda_max

n_signal = signal_mask.sum()
n_noise  = noise_mask.sum()

print(f'Signal eigenvalues (λ > {lambda_max:.4f}): {n_signal}')
print(f'Noise eigenvalues  (λ ≤ {lambda_max:.4f}): {n_noise}')
print()
print(f'Signal eigenvalues:')
for i, ev in enumerate(eigenvalues[signal_mask]):
    pct = ev / eigenvalues.sum() * 100
    print(f'  λ{i+1}: {ev:.4f} ({pct:.1f}% variance explained)')

print()
variance_signal = eigenvalues[signal_mask].sum() / eigenvalues.sum() * 100
variance_noise  = eigenvalues[noise_mask].sum()  / eigenvalues.sum() * 100
print(f'Variance in signal: {variance_signal:.1f}%')
print(f'Variance in noise:  {variance_noise:.1f}%')

=== Step 3: Marchenko-Pastur Fitting ===

Fitted sigma²: 0.0100
MP λ_min: 0.0001
MP λ_max: 0.0001

Signal eigenvalues (λ > 0.0001): 25
Noise eigenvalues  (λ ≤ 0.0001): 1

Signal eigenvalues:
  λ1: 5.3537 (20.6% variance explained)
  λ2: 3.5619 (13.7% variance explained)
  λ3: 2.5489 (9.8% variance explained)
  λ4: 1.8587 (7.1% variance explained)
  λ5: 1.6768 (6.4% variance explained)
  λ6: 1.6031 (6.2% variance explained)
  λ7: 1.4172 (5.5% variance explained)
  λ8: 1.3306 (5.1% variance explained)
  λ9: 1.0439 (4.0% variance explained)
  λ10: 0.9719 (3.7% variance explained)
  λ11: 0.7420 (2.9% variance explained)
  λ12: 0.6551 (2.5% variance explained)
  λ13: 0.5683 (2.2% variance explained)
  λ14: 0.4864 (1.9% variance explained)
  λ15: 0.3940 (1.5% variance explained)
  λ16: 0.3430 (1.3% variance explained)
  λ17: 0.3032 (1.2% variance explained)
  λ18: 0.2599 (1.0% variance explained)
  λ19: 0.2245 (0.9% variance explained)
  λ20: 0.2019 (0.8% variance explained)
  λ21: 0.1776 (0

In [46]:
# ── Fix: use sigma=1.0 directly ───────────────────────────────────
print('=== Marchenko-Pastur (fixed sigma=1.0) ===')
print()

# With standardized features sigma=1.0 always
sigma   = 1.0
q       = T / N  # 2329.9

lambda_min = sigma**2 * (1 - 1/np.sqrt(q))**2
lambda_max = sigma**2 * (1 + 1/np.sqrt(q))**2

print(f'q = T/N = {q:.1f}')
print(f'sigma = {sigma}')
print(f'MP λ_min = {lambda_min:.6f}')
print(f'MP λ_max = {lambda_max:.6f}')
print()

# At q=2329: λ_max ≈ 1.041
# Eigenvalues > 1.041 are SIGNAL
# Eigenvalues ≤ 1.041 are NOISE

signal_mask = eigenvalues > lambda_max
noise_mask  = ~signal_mask

n_signal = signal_mask.sum()
n_noise  = noise_mask.sum()

print(f'Signal eigenvalues (λ > {lambda_max:.4f}): {n_signal}')
print(f'Noise eigenvalues  (λ ≤ {lambda_max:.4f}): {n_noise}')
print()

print('Signal eigenvalues (real structure):')
for i, ev in enumerate(eigenvalues[signal_mask]):
    pct  = ev / eigenvalues.sum() * 100
    feat = valid_features[np.argmax(np.abs(eigenvectors[:, i]))]
    print(f'  λ{i+1}: {ev:.4f} ({pct:.1f}%)  dominant: {feat}')

print()
print('Noise eigenvalues (random, to be removed):')
for i, ev in enumerate(eigenvalues[noise_mask]):
    print(f'  λ{n_signal+i+1}: {ev:.4f} '
          f'({ev/eigenvalues.sum()*100:.1f}%)')

print()
print(f'Variance explained by signal: '
      f'{eigenvalues[signal_mask].sum()/eigenvalues.sum()*100:.1f}%')
print(f'Variance explained by noise:  '
      f'{eigenvalues[noise_mask].sum()/eigenvalues.sum()*100:.1f}%')

=== Marchenko-Pastur (fixed sigma=1.0) ===

q = T/N = 2329.9
sigma = 1.0
MP λ_min = 0.958995
MP λ_max = 1.041863

Signal eigenvalues (λ > 1.0419): 9
Noise eigenvalues  (λ ≤ 1.0419): 17

Signal eigenvalues (real structure):
  λ1: 5.3537 (20.6%)  dominant: vwap_dev
  λ2: 3.5619 (13.7%)  dominant: days_to_expiry
  λ3: 2.5489 (9.8%)  dominant: days_to_rbi
  λ4: 1.8587 (7.1%)  dominant: depth_imb_1
  λ5: 1.6768 (6.4%)  dominant: rv_trend
  λ6: 1.6031 (6.2%)  dominant: autocorr_600
  λ7: 1.4172 (5.5%)  dominant: spread_bps
  λ8: 1.3306 (5.1%)  dominant: cancel_z_30
  λ9: 1.0439 (4.0%)  dominant: mom_l_vl

Noise eigenvalues (random, to be removed):
  λ10: 0.9719 (3.7%)
  λ11: 0.7420 (2.9%)
  λ12: 0.6551 (2.5%)
  λ13: 0.5683 (2.2%)
  λ14: 0.4864 (1.9%)
  λ15: 0.3940 (1.5%)
  λ16: 0.3430 (1.3%)
  λ17: 0.3032 (1.2%)
  λ18: 0.2599 (1.0%)
  λ19: 0.2245 (0.9%)
  λ20: 0.2019 (0.8%)
  λ21: 0.1776 (0.7%)
  λ22: 0.1265 (0.5%)
  λ23: 0.0776 (0.3%)
  λ24: 0.0530 (0.2%)
  λ25: 0.0201 (0.1%)
  λ26: 0.0000 

In [47]:
# ── Step 4: Denoise correlation matrix ────────────────────────────
print('=== Step 4: Denoised Correlation Matrix ===')
print()

def denoise_correlation(eigenvalues, eigenvectors, 
                        n_signal, method='targeted_shrinkage'):
    """
    Lopez de Prado denoising methods:
    
    1. constant_residual: set noise eigenvalues to constant
    2. targeted_shrinkage: shrink noise toward average
    3. spectral: zero out noise eigenvalues
    """
    # Copy eigenvalues
    evals_denoised = eigenvalues.copy()
    
    if method == 'constant_residual':
        # Set noise eigenvalues to average of noise eigenvalues
        # Preserves trace (sum of eigenvalues = N)
        noise_mean = eigenvalues[n_signal:].mean()
        evals_denoised[n_signal:] = noise_mean
        
    elif method == 'targeted_shrinkage':
        # Shrink noise eigenvalues toward 1.0
        # (expected value under random matrix theory)
        alpha = 0.5  # shrinkage intensity
        evals_denoised[n_signal:] = (
            alpha * 1.0 + 
            (1-alpha) * eigenvalues[n_signal:].mean()
        )
        
    elif method == 'spectral':
        # Zero out noise eigenvalues completely
        evals_denoised[n_signal:] = 0
    
    # Rescale to preserve trace
    evals_denoised = evals_denoised * (
        eigenvalues.sum() / evals_denoised.sum()
    )
    
    # Reconstruct correlation matrix
    # C_denoised = W × diag(λ_denoised) × W^T
    corr_denoised = eigenvectors @ np.diag(evals_denoised) @ eigenvectors.T
    
    # Force diagonal = 1 (correlation matrix property)
    diag = np.diag(corr_denoised)
    corr_denoised = corr_denoised / np.outer(np.sqrt(diag), np.sqrt(diag))
    
    return corr_denoised, evals_denoised

# Apply denoising
n_signal = signal_mask.sum()  # 9

corr_denoised, evals_denoised = denoise_correlation(
    eigenvalues, eigenvectors, n_signal,
    method='constant_residual'
)

print(f'Original correlation matrix:')
print(f'  Mean abs off-diag: {np.abs(corr - np.eye(N)).mean():.4f}')
print(f'  Max abs off-diag:  {np.abs(corr - np.eye(N)).max():.4f}')
print()
print(f'Denoised correlation matrix:')
print(f'  Mean abs off-diag: {np.abs(corr_denoised - np.eye(N)).mean():.4f}')
print(f'  Max abs off-diag:  {np.abs(corr_denoised - np.eye(N)).max():.4f}')

# Compare eigenvalue spectra
print()
print('=== Eigenvalue Comparison ===')
print(f'{"#":<4} {"Original":>10} {"Denoised":>10} {"Change":>10}')
print('-'*36)
for i in range(N):
    orig = eigenvalues[i]
    den  = evals_denoised[i]
    chg  = den - orig
    marker = ' ← SIGNAL' if i < n_signal else ' (noise)'
    print(f'{i+1:<4} {orig:>10.4f} {den:>10.4f} {chg:>+10.4f}{marker}')

# ── Step 5: Build denoised features ───────────────────────────────
print()
print('=== Step 5: Denoised Feature Matrix ===')

# Project features onto signal eigenvectors only
# This gives us DENOISED features
signal_eigenvectors = eigenvectors[:, :n_signal]  # 26×9

# Denoised X = X × W_signal × W_signal^T
X_denoised = X_std @ signal_eigenvectors @ signal_eigenvectors.T

print(f'Original feature matrix: {X_std.shape}')
print(f'Signal eigenvectors:     {signal_eigenvectors.shape}')
print(f'Denoised feature matrix: {X_denoised.shape}')
print()

# Check: do denoised features retain IC?
fwd_1800 = df_sig['fwd_1800'].values

print('=== IC Comparison: Original vs Denoised ===')
print(f'{"Feature":<25} {"IC_orig":>10} {"IC_denoised":>12} {"Change":>8}')
print('-'*57)

ic_improvements = []
for i, f in enumerate(valid_features):
    ic_orig = pd.Series(X_std[:, i]).corr(
               pd.Series(fwd_1800[:len(X_std)]))
    ic_den  = pd.Series(X_denoised[:, i]).corr(
               pd.Series(fwd_1800[:len(X_denoised)]))
    chg     = ic_den - ic_orig
    ic_improvements.append({'feature': f,
                             'ic_orig': ic_orig,
                             'ic_den':  ic_den,
                             'chg':     chg})
    if abs(ic_orig) > 0.05 or abs(ic_den) > 0.05:
        print(f'{f:<25} {ic_orig:>10.4f} {ic_den:>12.4f} {chg:>+8.4f}')

df_ic = pd.DataFrame(ic_improvements)
print()
print(f'Avg IC orig:     {df_ic["ic_orig"].abs().mean():.4f}')
print(f'Avg IC denoised: {df_ic["ic_den"].abs().mean():.4f}')
print(f'Improvement:     {(df_ic["ic_den"].abs().mean() - df_ic["ic_orig"].abs().mean())/df_ic["ic_orig"].abs().mean()*100:.1f}%')

=== Step 4: Denoised Correlation Matrix ===

Original correlation matrix:
  Mean abs off-diag: 0.1465
  Max abs off-diag:  1.0000

Denoised correlation matrix:
  Mean abs off-diag: 0.1475
  Max abs off-diag:  0.7256

=== Eigenvalue Comparison ===
#      Original   Denoised     Change
------------------------------------
1        5.3537     5.3537    -0.0000 ← SIGNAL
2        3.5619     3.5619    -0.0000 ← SIGNAL
3        2.5489     2.5489    -0.0000 ← SIGNAL
4        1.8587     1.8587    -0.0000 ← SIGNAL
5        1.6768     1.6768    -0.0000 ← SIGNAL
6        1.6031     1.6031    -0.0000 ← SIGNAL
7        1.4172     1.4172    -0.0000 ← SIGNAL
8        1.3306     1.3306    -0.0000 ← SIGNAL
9        1.0439     1.0439    -0.0000 ← SIGNAL
10       0.9719     0.3297    -0.6422 (noise)
11       0.7420     0.3297    -0.4123 (noise)
12       0.6551     0.3297    -0.3254 (noise)
13       0.5683     0.3297    -0.2386 (noise)
14       0.4864     0.3297    -0.1567 (noise)
15       0.3940     0.329

In [48]:
# ── Step 6: ML on denoised features ───────────────────────────────
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
print('=== Step 6: Denoised vs Original Features ===')
print()

TARGET = 'fwd_1800'
dates  = sorted(df_sig['date'].unique())

# Add denoised features to df_sig
for i, f in enumerate(valid_features):
    df_sig[f'den_{f}'] = X_denoised[:, i]

den_features = [f'den_{f}' for f in valid_features]

# Walk-forward comparison
results_orig = []
results_den  = []

print(f'{"Test date":<14} {"IC_orig":>10} {"IC_den":>10} {"Improvement":>12}')
print('-'*48)

for test_idx in range(2, len(dates)):
    train_d = dates[:test_idx]
    test_d  = dates[test_idx]
    
    train = df_sig[df_sig['date'].isin(train_d)].dropna(
        subset=valid_features+den_features+[TARGET])
    test  = df_sig[df_sig['date']==test_d].dropna(
        subset=valid_features+den_features+[TARGET])
    
    if len(train) < 500 or len(test) < 100:
        continue
    
    y_train = train[TARGET].values
    y_test  = test[TARGET].values
    
    # Original features
    sc1      = StandardScaler()
    X_tr1    = sc1.fit_transform(train[valid_features].values)
    X_te1    = sc1.transform(test[valid_features].values)
    m1       = RidgeCV(alphas=[0.1,1,10,100,1000])
    m1.fit(X_tr1, y_train)
    ic_orig  = pd.Series(m1.predict(X_te1)).corr(pd.Series(y_test))
    
    # Denoised features
    sc2      = StandardScaler()
    X_tr2    = sc2.fit_transform(train[den_features].values)
    X_te2    = sc2.transform(test[den_features].values)
    m2       = RidgeCV(alphas=[0.1,1,10,100,1000])
    m2.fit(X_tr2, y_train)
    ic_den   = pd.Series(m2.predict(X_te2)).corr(pd.Series(y_test))
    
    improvement = ic_den - ic_orig
    results_orig.append(ic_orig)
    results_den.append(ic_den)
    
    print(f'{test_d:<14} {ic_orig:>10.4f} {ic_den:>10.4f} '
          f'{improvement:>+12.4f}')

print()
print(f'Avg OOS IC original:  {np.mean(results_orig):.4f}')
print(f'Avg OOS IC denoised:  {np.mean(results_den):.4f}')
print(f'Avg improvement:      {np.mean(results_den)-np.mean(results_orig):+.4f}')
print(f'Positive days orig:   {(np.array(results_orig)>0).mean()*100:.0f}%')
print(f'Positive days den:    {(np.array(results_den)>0).mean()*100:.0f}%')

=== Step 6: Denoised vs Original Features ===

Test date         IC_orig     IC_den  Improvement
------------------------------------------------
2026-09-24         0.1551     0.0850      -0.0701
2026-09-25        -0.0068    -0.0181      -0.0113
2026-09-30         0.2904     0.0566      -0.2338
2026-10-01         0.4010     0.3887      -0.0124

Avg OOS IC original:  0.2099
Avg OOS IC denoised:  0.1280
Avg improvement:      -0.0819
Positive days orig:   75%
Positive days den:    75%
